In [1]:
from pathlib import Path

# 현재 작업 폴더 확인
print("현재 작업 폴더:")
print(Path.cwd())

print("\n분석파일 후보:")
for p in sorted(Path(".").glob("*.pkl")):
    print("-", p.name)

현재 작업 폴더:
c:\Users\campus4D055\Desktop\통계프로젝트결측치파이썬보간계산이야호

분석파일 후보:
- actual_gap_info_locked.pkl
- actual_month_grid_locked.pkl
- full_gold_candidate_imputed.pkl
- full_gold_judgement_result.pkl
- full_gold_judgement_summary.pkl
- im_analysis_v1.pkl
- im_work.pkl
- im_work_imputed.pkl
- imputation_log.pkl
- industry_all_methods_robustness.pkl
- industry_all_repeat_wape.pkl
- industry_final_method_map.pkl
- industry_imputation_rows.pkl
- industry_imputation_windows.pkl
- industry_one_missing_final.pkl
- industry_one_missing_repeat.pkl
- industry_one_missing_summary.pkl
- industry_seasonal_repeat_wape.pkl
- industry_seasonal_robustness.pkl
- industry_unstable_extra10.pkl
- industry_wape.pkl
- industry_wape_detail.pkl
- industry_winners.pkl
- judgement_common_episodes.pkl
- judgement_common_masked.pkl
- judgement_common_with_prev.pkl
- judgement_imputed_missing.pkl
- judgement_preservation_masked.pkl
- judgement_preservation_result.pkl
- judgement_preservation_summary.pkl
- judgement_pre

In [3]:
import pandas as pd
import numpy as np

FILE_PATH = "im_analysis_v1.pkl"

df = pd.read_pickle(FILE_PATH)

print("사용 파일:", FILE_PATH)
print("행 수:", len(df))
print("열 수:", len(df.columns))

print("\n컬럼명:")
print(df.columns.tolist())

사용 파일: im_analysis_v1.pkl
행 수: 385756
열 수: 29

컬럼명:
['법인ID', '기준월', '업종_대분류', '업종_중분류', '사업장_시도', '사업장_시군구', '예금잔액합계', '투자상품잔액', '운전자금대출잔액', '시설자금대출잔액', '카드사용액', '거래금액합계', '요구불입금금액', '요구불출금금액', '원본결측월', '결측구간번호', '결측개월수', '원본결측유형', '보간된변수수', '보간발생', '3개월이상결측구간', '예금잔액합계_보간여부', '투자상품잔액_보간여부', '운전자금대출잔액_보간여부', '시설자금대출잔액_보간여부', '카드사용액_보간여부', '거래금액합계_보간여부', '요구불입금금액_보간여부', '요구불출금금액_보간여부']


In [4]:
# 08_계절조정_실험 - 표본 구조 확인

print("기준월 dtype:", df["기준월"].dtype)
print("요구불입금금액 dtype:", df["요구불입금금액"].dtype)
print("보간여부 dtype:", df["요구불입금금액_보간여부"].dtype)

print("\n기준월 범위:")
print("최소:", df["기준월"].min())
print("최대:", df["기준월"].max())

print("\n전체 법인 수:")
print(df["법인ID"].nunique())

print("\n요구불입금금액 결측 여부:")
print(df["요구불입금금액"].isna().value_counts(dropna=False))

print("\n요구불입금금액_보간여부 값:")
print(df["요구불입금금액_보간여부"].value_counts(dropna=False))

기준월 dtype: datetime64[us]
요구불입금금액 dtype: float64
보간여부 dtype: bool

기준월 범위:
최소: 2023-01-01 00:00:00
최대: 2025-12-01 00:00:00

전체 법인 수:
15473

요구불입금금액 결측 여부:
요구불입금금액
False    366119
True      19637
Name: count, dtype: int64

요구불입금금액_보간여부 값:
요구불입금금액_보간여부
False    385625
True        131
Name: count, dtype: int64


In [5]:
# ============================================================
# 08 계절조정 실험
# STEP 1. 요구불입금 실제 관측값만 분석 대상으로 정의
# ============================================================

season_base = df.loc[
    df["요구불입금금액"].notna()
    & (~df["요구불입금금액_보간여부"])
].copy()

print("실제 관측값 행 수:", len(season_base))
print("실제 관측 기업 수:", season_base["법인ID"].nunique())

# 기업-월 중복 여부 확인
dup_count = season_base.duplicated(
    subset=["법인ID", "기준월"]
).sum()

print("기업-월 중복 행 수:", dup_count)

# 기업별 실제 관측 개월 수
firm_obs = (
    season_base.groupby("법인ID")["기준월"]
    .nunique()
)

print("\n기업별 관측개월 수 요약:")
print(firm_obs.describe())

print("\n관측개월 수 구간별 기업 수:")
print(pd.cut(
    firm_obs,
    bins=[0, 11, 23, 29, 35, 36],
    labels=["1~11개월", "12~23개월", "24~29개월", "30~35개월", "36개월"],
    include_lowest=True
).value_counts().sort_index())

실제 관측값 행 수: 365988
실제 관측 기업 수: 15473
기업-월 중복 행 수: 0

기업별 관측개월 수 요약:
count    15473.000000
mean        23.653332
std         11.890890
min          1.000000
25%         13.000000
50%         27.000000
75%         35.000000
max         36.000000
Name: 기준월, dtype: float64

관측개월 수 구간별 기업 수:
기준월
1~11개월     3393
12~23개월    3459
24~29개월    1544
30~35개월    3705
36개월       3372
Name: count, dtype: int64


In [6]:
# ============================================================
# STEP 2. 36개월 완전관측 기업의 연도별 월 패턴 반복성 확인
# ============================================================

# 36개월 모두 실제 관측된 기업
complete_ids = firm_obs[firm_obs == 36].index

season_36 = season_base.loc[
    season_base["법인ID"].isin(complete_ids),
    ["법인ID", "기준월", "업종_대분류", "요구불입금금액"]
].copy()

season_36["연도"] = season_36["기준월"].dt.year
season_36["월"] = season_36["기준월"].dt.month

print("36개월 완전관측 기업 수:", season_36["법인ID"].nunique())
print("분석 행 수:", len(season_36))


# 기업 × 연도별로 1~12월을 가로로 펼침
monthly = season_36.pivot(
    index=["법인ID", "연도"],
    columns="월",
    values="요구불입금금액"
)

print("\n기업×연도 행 수:", len(monthly))
print("남아 있는 결측값 수:", monthly.isna().sum().sum())


# 연중 12개월 값이 모두 똑같은 기업-연도 확인
constant_year = monthly.nunique(axis=1) <= 1

print("\n12개월 값이 모두 동일한 기업-연도 수:")
print(
    constant_year
    .groupby(level="연도")
    .sum()
)


# ------------------------------------------------------------
# 기업별 연도 간 Spearman 상관계수
# ------------------------------------------------------------

results = []

for firm_id in complete_ids:

    y2023 = monthly.loc[(firm_id, 2023)]
    y2024 = monthly.loc[(firm_id, 2024)]
    y2025 = monthly.loc[(firm_id, 2025)]

    corr_23_24 = y2023.corr(y2024, method="spearman")
    corr_24_25 = y2024.corr(y2025, method="spearman")
    corr_23_25 = y2023.corr(y2025, method="spearman")

    correlations = pd.Series([
        corr_23_24,
        corr_24_25,
        corr_23_25
    ])

    results.append({
        "법인ID": firm_id,
        "corr_2023_2024": corr_23_24,
        "corr_2024_2025": corr_24_25,
        "corr_2023_2025": corr_23_25,
        "반복성_중앙값": correlations.median()
    })


firm_repeat = pd.DataFrame(results)


# 개별 법인ID는 출력하지 않고 분포만 확인
print("\n연도 간 월 패턴 상관계수 분포:")
print(
    firm_repeat[
        [
            "corr_2023_2024",
            "corr_2024_2025",
            "corr_2023_2025",
            "반복성_중앙값"
        ]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\n반복성 중앙값을 계산할 수 없는 기업 수:")
print(firm_repeat["반복성_중앙값"].isna().sum())

36개월 완전관측 기업 수: 3372
분석 행 수: 121392

기업×연도 행 수: 10116
남아 있는 결측값 수: 0

12개월 값이 모두 동일한 기업-연도 수:
연도
2023    25
2024    30
2025    29
dtype: int64


c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\panda


연도 간 월 패턴 상관계수 분포:
       corr_2023_2024  corr_2024_2025  corr_2023_2025      반복성_중앙값
count     3333.000000     3327.000000     3333.000000  3351.000000
mean         0.094388        0.143660        0.084974     0.083333
std          0.335814        0.333263        0.337535     0.260643
min         -0.845766       -0.839161       -0.832746    -0.758388
10%         -0.341802       -0.304993       -0.349218    -0.238702
25%         -0.141264       -0.094737       -0.152213    -0.097176
50%          0.094571        0.148940        0.077330     0.067020
75%          0.332749        0.385823        0.328671     0.254758
90%          0.537352        0.588856        0.539217     0.424561
max          1.000000        1.000000        0.982633     1.000000

반복성 중앙값을 계산할 수 없는 기업 수:
21


c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [7]:
# ============================================================
# STEP 3. 기업 규모를 제거한 뒤 전체 공통 월 패턴 확인
# ============================================================

rank_base = season_36[
    ["법인ID", "연도", "월", "요구불입금금액"]
].copy()

# 기업×연도 안에서 12개월 금액의 순위를 계산
# 1 = 가장 낮은 달
# 12 = 가장 높은 달
rank_base["월순위"] = (
    rank_base
    .groupby(["법인ID", "연도"])["요구불입금금액"]
    .rank(method="average")
)

# 12개월 값이 모두 동일한 기업-연도는 정보가 없으므로 제외
valid_years = monthly.index[~constant_year]

valid_index = pd.MultiIndex.from_frame(
    rank_base[["법인ID", "연도"]]
)

rank_base = rank_base.loc[
    valid_index.isin(valid_years)
].copy()


# ------------------------------------------------------------
# 연도 × 월별 기업들의 월순위 집계
# ------------------------------------------------------------

month_pattern = (
    rank_base
    .groupby(["연도", "월"])["월순위"]
    .agg(
        기업수="count",
        평균순위="mean",
        중앙순위="median",
        표준편차="std"
    )
    .reset_index()
)

print("연도별 월순위:")
print(
    month_pattern.pivot(
        index="월",
        columns="연도",
        values="평균순위"
    ).round(3)
)


# ------------------------------------------------------------
# 3개 연도를 합친 월별 공통 패턴
# ------------------------------------------------------------

overall_month = (
    rank_base
    .groupby("월")["월순위"]
    .agg(
        기업연도수="count",
        평균순위="mean",
        중앙순위="median",
        표준편차="std"
    )
)

print("\n3개 연도 통합 월별 순위:")
print(overall_month.round(3))


# ------------------------------------------------------------
# 연도별 12개월 패턴이 서로 닮았는지도 확인
# ------------------------------------------------------------

year_month_profile = (
    month_pattern
    .pivot(
        index="월",
        columns="연도",
        values="평균순위"
    )
)

print("\n연도별 공통 월패턴 간 Spearman 상관:")
print(year_month_profile.corr(method="spearman").round(3))

연도별 월순위:
연도   2023   2024   2025
월                      
1   6.441  6.813  6.308
2   5.668  6.285  6.178
3   6.717  5.801  6.495
4   5.905  7.093  6.910
5   7.009  6.744  6.205
6   6.760  5.994  6.766
7   6.343  7.061  6.668
8   6.365  5.937  5.705
9   6.295  6.508  7.122
10  6.519  6.226  6.314
11  6.660  5.904  5.589
12  7.319  7.635  7.741

3개 연도 통합 월별 순위:
    기업연도수   평균순위  중앙순위   표준편차
월                            
1   10032  6.520  6.50  3.543
2   10032  6.044  6.00  3.457
3   10032  6.338  6.00  3.416
4   10032  6.635  6.75  3.321
5   10032  6.653  6.50  3.243
6   10032  6.507  6.50  3.282
7   10032  6.690  7.00  3.240
8   10032  6.003  6.00  3.245
9   10032  6.641  6.50  3.305
10  10032  6.353  6.00  3.325
11  10032  6.051  6.00  3.383
12  10032  7.565  8.00  3.563

연도별 공통 월패턴 간 Spearman 상관:
연도     2023   2024   2025
연도                       
2023  1.000 -0.119  0.084
2024 -0.119  1.000  0.559
2025  0.084  0.559  1.000


In [8]:
# ============================================================
# STEP 4. 기업 규모를 제거한 월별 상대금액의 크기 확인
# ============================================================

relative_base = season_36[
    ["법인ID", "연도", "월", "요구불입금금액"]
].copy()

# 기업×연도의 12개월 중앙값
firm_year_median = (
    relative_base
    .groupby(["법인ID", "연도"])["요구불입금금액"]
    .transform("median")
)

relative_base["연간중앙값"] = firm_year_median

# 중앙값이 0보다 큰 기업-연도만 비율 계산 가능
usable = relative_base["연간중앙값"] > 0

print("전체 기업-연도 수:",
      relative_base[["법인ID", "연도"]].drop_duplicates().shape[0])

print("연간 중앙값 > 0인 기업-연도 수:",
      relative_base.loc[usable, ["법인ID", "연도"]].drop_duplicates().shape[0])

print("연간 중앙값 = 0인 기업-연도 수:",
      relative_base.loc[~usable, ["법인ID", "연도"]].drop_duplicates().shape[0])


# 기업 자신의 연간 중앙값 대비 상대금액
relative_base = relative_base.loc[usable].copy()

relative_base["상대금액"] = (
    relative_base["요구불입금금액"]
    / relative_base["연간중앙값"]
)


# ------------------------------------------------------------
# 연도 × 월별 중앙 상대금액
# 평균이 아니라 중앙값을 우선 사용
# ------------------------------------------------------------

relative_month = (
    relative_base
    .groupby(["연도", "월"])["상대금액"]
    .agg(
        기업수="count",
        중앙상대금액="median",
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

print("\n연도별 월 중앙 상대금액:")
print(
    relative_month.pivot(
        index="월",
        columns="연도",
        values="중앙상대금액"
    ).round(3)
)


# ------------------------------------------------------------
# 3개 연도 통합
# ------------------------------------------------------------

overall_relative = (
    relative_base
    .groupby("월")["상대금액"]
    .agg(
        기업연도수="count",
        중앙상대금액="median",
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75)
    )
)

print("\n3개 연도 통합 월별 상대금액:")
print(overall_relative.round(3))

전체 기업-연도 수: 10116
연간 중앙값 > 0인 기업-연도 수: 9778
연간 중앙값 = 0인 기업-연도 수: 338

연도별 월 중앙 상대금액:
연도   2023   2024   2025
월                      
1   1.000  1.017  1.000
2   0.944  1.000  0.995
3   1.000  0.957  1.000
4   0.969  1.030  1.018
5   1.024  1.000  1.000
6   1.000  0.973  1.000
7   1.000  1.034  1.000
8   1.000  0.979  0.952
9   1.000  1.000  1.037
10  1.000  0.991  1.000
11  1.000  0.970  0.940
12  1.054  1.089  1.121

3개 연도 통합 월별 상대금액:
    기업연도수  중앙상대금액    Q25    Q75
월                              
1    9778   1.000  0.783  1.303
2    9778   0.980  0.731  1.176
3    9778   1.000  0.784  1.232
4    9778   1.000  0.828  1.257
5    9778   1.000  0.830  1.238
6    9778   1.000  0.835  1.245
7    9778   1.000  0.846  1.254
8    9778   0.979  0.757  1.143
9    9778   1.000  0.833  1.263
10   9778   1.000  0.784  1.210
11   9778   0.981  0.750  1.177
12   9778   1.086  0.909  1.661


In [9]:
# ============================================================
# STEP 5. 업종별 월 패턴이 3년간 반복되는지 확인
# ============================================================

industry_base = season_36[
    ["법인ID", "연도", "월", "업종_대분류", "요구불입금금액"]
].copy()

# 기업×연도 중앙값
industry_base["연간중앙값"] = (
    industry_base
    .groupby(["법인ID", "연도"])["요구불입금금액"]
    .transform("median")
)

# 중앙값이 0보다 큰 기업-연도만 사용
industry_base = industry_base.loc[
    industry_base["연간중앙값"] > 0
].copy()

industry_base["상대금액"] = (
    industry_base["요구불입금금액"]
    / industry_base["연간중앙값"]
)


# ------------------------------------------------------------
# 업종별 표본 크기
# ------------------------------------------------------------

industry_size = (
    industry_base
    .groupby("업종_대분류")["법인ID"]
    .nunique()
    .sort_values(ascending=False)
)

print("업종별 기업 수:")
print(industry_size)


# ------------------------------------------------------------
# 업종 × 연도 × 월 중앙 상대금액
# ------------------------------------------------------------

industry_month = (
    industry_base
    .groupby(["업종_대분류", "연도", "월"])["상대금액"]
    .agg(
        기업수="count",
        중앙상대금액="median"
    )
    .reset_index()
)


# ------------------------------------------------------------
# 업종별로 2023/2024/2025의 12개월 패턴 상관계수 계산
# ------------------------------------------------------------

industry_repeat = []

for industry, g in industry_month.groupby("업종_대분류"):

    pivot = g.pivot(
        index="월",
        columns="연도",
        values="중앙상대금액"
    )

    # 세 연도가 모두 존재하는 경우만
    if all(y in pivot.columns for y in [2023, 2024, 2025]):

        c23_24 = pivot[2023].corr(pivot[2024], method="spearman")
        c24_25 = pivot[2024].corr(pivot[2025], method="spearman")
        c23_25 = pivot[2023].corr(pivot[2025], method="spearman")

        industry_repeat.append({
            "업종_대분류": industry,
            "기업수": industry_size.get(industry, np.nan),
            "corr_2023_2024": c23_24,
            "corr_2024_2025": c24_25,
            "corr_2023_2025": c23_25,
            "반복성_중앙값": pd.Series(
                [c23_24, c24_25, c23_25]
            ).median()
        })

industry_repeat = pd.DataFrame(industry_repeat)

print("\n업종별 월 패턴 반복성:")
print(
    industry_repeat
    .sort_values("반복성_중앙값", ascending=False)
    .round(3)
    .to_string(index=False)
)

업종별 기업 수:
업종_대분류
제조업                         1073
도매 및 소매업                     689
건설업                          570
부동산업                         260
운수 및 창고업                     141
사업시설 관리, 사업 지원 및 임대 서비스업     105
전문, 과학 및 기술 서비스업              82
협회 및 단체, 수리 및 기타 개인 서비스업      79
숙박 및 음식점업                     59
수도, 하수 및 폐기물 처리, 원료 재생업       54
정보통신업                         53
교육 서비스업                       31
예술, 스포츠 및 여가관련 서비스업           28
전기, 가스, 증기 및 공기조절 공급업         25
금융 및 보험업                      22
보건업 및 사회복지 서비스업               19
광업                             9
농업, 임업 및 어업                    5
Name: 법인ID, dtype: int64

업종별 월 패턴 반복성:
                  업종_대분류  기업수  corr_2023_2024  corr_2024_2025  corr_2023_2025  반복성_중앙값
                     건설업  570           0.681           0.844           0.645    0.681
                운수 및 창고업  141           0.520           0.517          -0.153    0.517
   전기, 가스, 증기 및 공기조절 공급업   25           0.515           0.491           0.542    0.515
협

c:\Users\campus4D055\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [10]:
# ============================================================
# STEP 6. 주요 업종의 실제 월별 상대금액 패턴 확인
# ============================================================

major_industries = industry_size[
    industry_size >= 100
].index

major_pattern = (
    industry_month[
        industry_month["업종_대분류"].isin(major_industries)
    ]
    .pivot_table(
        index=["업종_대분류", "월"],
        columns="연도",
        values="중앙상대금액"
    )
    .round(3)
)

for industry in major_industries:
    
    print("\n" + "=" * 60)
    print(industry, "| 기업 수:", industry_size[industry])
    print("=" * 60)
    
    print(
        major_pattern.loc[industry]
    )


제조업 | 기업 수: 1073
연도   2023   2024   2025
월                      
1   1.000  1.021  1.000
2   0.929  1.000  0.987
3   1.000  0.940  1.000
4   0.978  1.034  1.032
5   1.042  1.010  0.983
6   1.000  0.969  1.032
7   1.000  1.053  1.000
8   1.000  0.960  0.936
9   0.994  1.000  1.055
10  1.000  0.978  1.000
11  1.000  0.955  0.919
12  1.043  1.087  1.088

도매 및 소매업 | 기업 수: 689
연도   2023   2024   2025
월                      
1   1.011  1.029  1.000
2   0.944  1.000  1.000
3   1.019  0.944  1.000
4   0.956  1.050  1.018
5   1.037  1.018  1.000
6   1.000  0.939  1.000
7   0.981  1.051  1.013
8   1.000  0.981  0.950
9   1.000  1.000  1.029
10  1.029  0.990  1.000
11  1.007  0.946  0.933
12  1.000  1.046  1.081

건설업 | 기업 수: 570
연도   2023   2024   2025
월                      
1   1.000  1.000  1.000
2   0.769  0.927  0.805
3   1.019  0.962  1.000
4   0.889  1.000  1.030
5   1.000  0.980  0.942
6   1.077  1.030  1.050
7   0.952  1.000  1.000
8   0.957  0.893  0.944
9   1.100  1.037  1.056
10  0.7

In [11]:
# ============================================================
# STEP 7. 12월 효과가 같은 기업에서 반복되는지 확인
# ============================================================

december = industry_base.loc[
    industry_base["월"] == 12,
    ["법인ID", "업종_대분류", "연도", "상대금액"]
].copy()

# 기업별 2023/2024/2025 12월 상대금액을 가로로 펼침
dec_firm = december.pivot_table(
    index=["법인ID", "업종_대분류"],
    columns="연도",
    values="상대금액"
).reset_index()

# 세 연도가 모두 있는 기업만
dec_firm = dec_firm.dropna(
    subset=[2023, 2024, 2025]
).copy()

# 같은 기업에서 12월 상승이 얼마나 반복되는지
dec_firm["3년모두_1초과"] = (
    (dec_firm[2023] > 1) &
    (dec_firm[2024] > 1) &
    (dec_firm[2025] > 1)
)

dec_firm["3년중2년이상_1초과"] = (
    (dec_firm[[2023, 2024, 2025]] > 1)
    .sum(axis=1) >= 2
)

# 비교적 큰 상승도 따로 확인
dec_firm["3년중2년이상_20퍼상승"] = (
    (dec_firm[[2023, 2024, 2025]] > 1.20)
    .sum(axis=1) >= 2
)

# 기업별 3년 12월 효과의 중앙값
dec_firm["12월_3년중앙값"] = (
    dec_firm[[2023, 2024, 2025]]
    .median(axis=1)
)


# ------------------------------------------------------------
# 전체 결과
# ------------------------------------------------------------

print("분석 기업 수:", len(dec_firm))

print("\n전체 기업:")
print(
    "3년 모두 12월 > 연간중앙값:",
    round(dec_firm["3년모두_1초과"].mean() * 100, 1), "%"
)

print(
    "3년 중 2년 이상 12월 > 연간중앙값:",
    round(dec_firm["3년중2년이상_1초과"].mean() * 100, 1), "%"
)

print(
    "3년 중 2년 이상 12월 20% 이상 상승:",
    round(dec_firm["3년중2년이상_20퍼상승"].mean() * 100, 1), "%"
)

print(
    "기업별 12월 3년 중앙효과의 중앙값:",
    round(dec_firm["12월_3년중앙값"].median(), 3)
)


# ------------------------------------------------------------
# 업종별 결과
# 표본 50개 이상 업종만 출력
# ------------------------------------------------------------

dec_industry = (
    dec_firm
    .groupby("업종_대분류")
    .agg(
        기업수=("법인ID", "count"),
        삼년모두_상승비율=("3년모두_1초과", "mean"),
        이년이상_상승비율=("3년중2년이상_1초과", "mean"),
        이년이상_20퍼상승비율=("3년중2년이상_20퍼상승", "mean"),
        기업별_12월효과_중앙값=("12월_3년중앙값", "median")
    )
    .reset_index()
)

dec_industry = dec_industry[
    dec_industry["기업수"] >= 50
].copy()

for col in [
    "삼년모두_상승비율",
    "이년이상_상승비율",
    "이년이상_20퍼상승비율"
]:
    dec_industry[col] *= 100

print("\n업종별 12월 반복효과:")
print(
    dec_industry
    .sort_values(
        "이년이상_상승비율",
        ascending=False
    )
    .round(1)
    .to_string(index=False)
)

분석 기업 수: 3206

전체 기업:
3년 모두 12월 > 연간중앙값: 27.8 %
3년 중 2년 이상 12월 > 연간중앙값: 60.8 %
3년 중 2년 이상 12월 20% 이상 상승: 38.6 %
기업별 12월 3년 중앙효과의 중앙값: 1.077

업종별 12월 반복효과:
                  업종_대분류  기업수  삼년모두_상승비율  이년이상_상승비율  이년이상_20퍼상승비율  기업별_12월효과_중앙값
                     건설업  550       48.5       80.7          70.2            1.7
        전문, 과학 및 기술 서비스업   78       44.9       70.5          50.0            1.2
                   정보통신업   51       43.1       66.7          47.1            1.1
협회 및 단체, 수리 및 기타 개인 서비스업   78       38.5       65.4          37.2            1.1
                     제조업 1047       23.9       59.3          36.3            1.1
                운수 및 창고업  139       19.4       59.0          26.6            1.0
 수도, 하수 및 폐기물 처리, 원료 재생업   51       29.4       56.9          35.3            1.1
                도매 및 소매업  672       20.8       55.8          25.7            1.0
사업시설 관리, 사업 지원 및 임대 서비스업  103       29.1       53.4          29.1            1.0
                    부동산업  246      

In [12]:
# ============================================================
# STEP 8. 모든 월에 대해 기업 수준의 반복 방향 확인
# ============================================================

month_firm = (
    industry_base[
        ["법인ID", "업종_대분류", "연도", "월", "상대금액"]
    ]
    .pivot_table(
        index=["법인ID", "업종_대분류", "월"],
        columns="연도",
        values="상대금액"
    )
    .reset_index()
)

# 3개 연도가 모두 존재하는 경우만
month_firm = month_firm.dropna(
    subset=[2023, 2024, 2025]
).copy()


# 3년 중 몇 년이 1보다 높은지 / 낮은지
vals = month_firm[[2023, 2024, 2025]]

month_firm["상승연도수"] = (vals > 1).sum(axis=1)
month_firm["하락연도수"] = (vals < 1).sum(axis=1)

month_firm["2년이상_상승"] = month_firm["상승연도수"] >= 2
month_firm["2년이상_하락"] = month_firm["하락연도수"] >= 2

month_firm["3년모두_상승"] = month_firm["상승연도수"] == 3
month_firm["3년모두_하락"] = month_firm["하락연도수"] == 3

# 기업별 해당 월의 3년 중앙 효과
month_firm["3년중앙효과"] = vals.median(axis=1)


# ------------------------------------------------------------
# 전체 기업 기준 월별 반복성
# ------------------------------------------------------------

overall_month_repeat = (
    month_firm
    .groupby("월")
    .agg(
        기업수=("법인ID", "count"),
        이년이상_상승비율=("2년이상_상승", "mean"),
        이년이상_하락비율=("2년이상_하락", "mean"),
        삼년모두_상승비율=("3년모두_상승", "mean"),
        삼년모두_하락비율=("3년모두_하락", "mean"),
        기업별_3년중앙효과=("3년중앙효과", "median")
    )
)

ratio_cols = [
    "이년이상_상승비율",
    "이년이상_하락비율",
    "삼년모두_상승비율",
    "삼년모두_하락비율"
]

overall_month_repeat[ratio_cols] *= 100

print("전체 기업 월별 반복방향:")
print(overall_month_repeat.round(1))


# ------------------------------------------------------------
# 업종별
# 표본 50개 이상 업종만
# ------------------------------------------------------------

industry_month_repeat = (
    month_firm
    .groupby(["업종_대분류", "월"])
    .agg(
        기업수=("법인ID", "count"),
        이년이상_상승비율=("2년이상_상승", "mean"),
        이년이상_하락비율=("2년이상_하락", "mean"),
        삼년모두_상승비율=("3년모두_상승", "mean"),
        삼년모두_하락비율=("3년모두_하락", "mean"),
        기업별_3년중앙효과=("3년중앙효과", "median")
    )
    .reset_index()
)

for col in ratio_cols:
    industry_month_repeat[col] *= 100


major_repeat = industry_month_repeat[
    industry_month_repeat["기업수"] >= 50
].copy()


# 우선 건설업만 자세히 출력
construction = major_repeat[
    major_repeat["업종_대분류"] == "건설업"
]

print("\n건설업 월별 반복방향:")
print(
    construction[
        [
            "월",
            "기업수",
            "이년이상_상승비율",
            "이년이상_하락비율",
            "삼년모두_상승비율",
            "삼년모두_하락비율",
            "기업별_3년중앙효과"
        ]
    ]
    .round(1)
    .to_string(index=False)
)

전체 기업 월별 반복방향:
     기업수  이년이상_상승비율  이년이상_하락비율  삼년모두_상승비율  삼년모두_하락비율  기업별_3년중앙효과
월                                                               
1   3206       47.4       43.0       14.2       12.9         1.0
2   3206       38.0       52.2        9.8       17.7         1.0
3   3206       41.2       48.0       10.6       14.2         1.0
4   3206       46.9       41.7       13.0       10.7         1.0
5   3206       47.1       41.6       13.0       11.4         1.0
6   3206       43.3       44.6       12.8       11.5         1.0
7   3206       48.5       39.4       14.7       10.9         1.0
8   3206       33.5       54.2        7.8       16.4         1.0
9   3206       46.8       42.0       13.7       10.0         1.0
10  3206       41.5       47.6       10.7       13.2         1.0
11  3206       37.0       52.4        9.5       16.0         1.0
12  3206       60.8       30.3       27.8        7.5         1.1

건설업 월별 반복방향:
 월  기업수  이년이상_상승비율  이년이상_하락비율  삼년모두_상승비율  삼년모두_하락비율  기업별_3년중앙

In [13]:
# ============================================================
# STEP 9. 건설업 계절패턴 표본확장 검증
# 36개월 완전관측 → 30개월 이상 실제관측 기업
# ============================================================

# 실제 관측 30개월 이상 기업
ids_30plus = firm_obs[firm_obs >= 30].index

construction_30 = season_base.loc[
    season_base["법인ID"].isin(ids_30plus)
    & (season_base["업종_대분류"] == "건설업"),
    ["법인ID", "기준월", "요구불입금금액"]
].copy()

construction_30["연도"] = construction_30["기준월"].dt.year
construction_30["월"] = construction_30["기준월"].dt.month


# ------------------------------------------------------------
# 기업×연도별 실제 관측월 수
# ------------------------------------------------------------

obs_year = (
    construction_30
    .groupby(["법인ID", "연도"])["월"]
    .nunique()
    .rename("관측월수")
    .reset_index()
)

construction_30 = construction_30.merge(
    obs_year,
    on=["법인ID", "연도"],
    how="left"
)

# 한 해에 실제 관측이 10개월 이상인 경우만 사용
construction_30 = construction_30[
    construction_30["관측월수"] >= 10
].copy()


# ------------------------------------------------------------
# 기업×연도 중앙값으로 정규화
# ------------------------------------------------------------

construction_30["연간중앙값"] = (
    construction_30
    .groupby(["법인ID", "연도"])["요구불입금금액"]
    .transform("median")
)

construction_30 = construction_30[
    construction_30["연간중앙값"] > 0
].copy()

construction_30["상대금액"] = (
    construction_30["요구불입금금액"]
    / construction_30["연간중앙값"]
)


# ------------------------------------------------------------
# 연도 × 월 중앙 상대금액
# ------------------------------------------------------------

construction_30_pattern = (
    construction_30
    .groupby(["연도", "월"])["상대금액"]
    .agg(
        기업수="count",
        중앙상대금액="median"
    )
    .reset_index()
)

print("분석에 포함된 건설업 기업 수:",
      construction_30["법인ID"].nunique())

print("\n30개월 이상 기업 - 건설업 월별 중앙 상대금액:")
print(
    construction_30_pattern
    .pivot(
        index="월",
        columns="연도",
        values="중앙상대금액"
    )
    .round(3)
)

분석에 포함된 건설업 기업 수: 1120

30개월 이상 기업 - 건설업 월별 중앙 상대금액:
연도   2023   2024   2025
월                      
1   1.000  1.000  1.000
2   0.833  0.932  0.809
3   1.022  0.966  1.000
4   0.957  1.000  1.029
5   1.000  1.000  0.980
6   1.062  1.022  1.043
7   0.959  1.000  1.000
8   0.957  0.927  0.950
9   1.111  1.020  1.033
10  0.793  0.941  0.977
11  1.000  0.954  0.852
12  1.562  1.600  1.650


In [14]:
# ============================================================
# STEP 9-A. 업종 계절성 유형 1차 분류
# 아직 계절지수는 만들지 않는다.
# ============================================================

# ------------------------------------------------------------
# 1. 업종 × 월별 기업 수준 반복성/효과 계산
# ------------------------------------------------------------

industry_month_evidence = (
    month_firm
    .groupby(["업종_대분류", "월"])
    .agg(
        기업수=("법인ID", "count"),
        이년이상_상승비율=("2년이상_상승", "mean"),
        이년이상_하락비율=("2년이상_하락", "mean"),
        삼년모두_상승비율=("3년모두_상승", "mean"),
        삼년모두_하락비율=("3년모두_하락", "mean"),
        삼년중앙효과=("3년중앙효과", "median")
    )
    .reset_index()
)

# 비율 → %
for col in [
    "이년이상_상승비율",
    "이년이상_하락비율",
    "삼년모두_상승비율",
    "삼년모두_하락비율"
]:
    industry_month_evidence[col] *= 100


# ------------------------------------------------------------
# 2. '계절패턴 월' 판정
# 상승: 60% 이상 반복 + 중앙효과 1.10 이상
# 하락: 60% 이상 반복 + 중앙효과 0.90 이하
# ------------------------------------------------------------

industry_month_evidence["상승패턴"] = (
    (industry_month_evidence["이년이상_상승비율"] >= 60)
    & (industry_month_evidence["삼년중앙효과"] >= 1.10)
)

industry_month_evidence["하락패턴"] = (
    (industry_month_evidence["이년이상_하락비율"] >= 60)
    & (industry_month_evidence["삼년중앙효과"] <= 0.90)
)

industry_month_evidence["패턴월"] = (
    industry_month_evidence["상승패턴"]
    | industry_month_evidence["하락패턴"]
)


# 방향 표시
industry_month_evidence["패턴방향"] = ""

industry_month_evidence.loc[
    industry_month_evidence["상승패턴"],
    "패턴방향"
] = "상승"

industry_month_evidence.loc[
    industry_month_evidence["하락패턴"],
    "패턴방향"
] = "하락"


# ------------------------------------------------------------
# 3. 업종별 패턴 월 개수와 해당 월 목록
# ------------------------------------------------------------

pattern_summary = (
    industry_month_evidence
    .groupby("업종_대분류")
    .agg(
        기업수=("기업수", "max"),
        패턴월수=("패턴월", "sum")
    )
    .reset_index()
)

pattern_month_list = (
    industry_month_evidence[
        industry_month_evidence["패턴월"]
    ]
    .assign(
        표시=lambda x:
        x["월"].astype(str) + "월(" + x["패턴방향"] + ")"
    )
    .groupby("업종_대분류")["표시"]
    .apply(", ".join)
    .rename("패턴월목록")
    .reset_index()
)

pattern_summary = pattern_summary.merge(
    pattern_month_list,
    on="업종_대분류",
    how="left"
)

pattern_summary["패턴월목록"] = (
    pattern_summary["패턴월목록"]
    .fillna("-")
)


# ------------------------------------------------------------
# 4. 앞서 계산한 업종 전체 12개월 반복성 결합
# ------------------------------------------------------------

pattern_summary = pattern_summary.merge(
    industry_repeat[
        ["업종_대분류", "반복성_중앙값"]
    ],
    on="업종_대분류",
    how="left"
)


# ------------------------------------------------------------
# 5. 사전 정의한 규칙으로 유형 분류
# ------------------------------------------------------------

def classify(row):

    if row["기업수"] < 50:
        return "판단보류_표본부족"

    if (
        row["패턴월수"] >= 3
        and pd.notna(row["반복성_중앙값"])
        and row["반복성_중앙값"] >= 0.50
    ):
        return "① 다월 계절형"

    if 1 <= row["패턴월수"] <= 2:
        return "② 특정월 계절형"

    if row["패턴월수"] == 0:
        return "③ 비계절형"

    return "판단보류_패턴불일치"


pattern_summary["1차분류"] = pattern_summary.apply(
    classify,
    axis=1
)


# ------------------------------------------------------------
# 6. 결과 출력
# ------------------------------------------------------------

print(
    pattern_summary[
        [
            "업종_대분류",
            "기업수",
            "패턴월수",
            "패턴월목록",
            "반복성_중앙값",
            "1차분류"
        ]
    ]
    .sort_values(
        ["1차분류", "기업수"],
        ascending=[True, False]
    )
    .round(3)
    .to_string(index=False)
)

                  업종_대분류  기업수  패턴월수                                                    패턴월목록  반복성_중앙값      1차분류
                     건설업  550     2                                          2월(하락), 12월(상승)    0.681 ② 특정월 계절형
        전문, 과학 및 기술 서비스업   78     1                                                  12월(상승)    0.290 ② 특정월 계절형
협회 및 단체, 수리 및 기타 개인 서비스업   78     1                                                  12월(상승)    0.460 ② 특정월 계절형
                   정보통신업   51     1                                                  12월(상승)    0.315 ② 특정월 계절형
                     제조업 1047     0                                                        -    0.125    ③ 비계절형
                도매 및 소매업  672     0                                                        -   -0.344    ③ 비계절형
                    부동산업  246     0                                                        -   -0.085    ③ 비계절형
                운수 및 창고업  139     0                                                        -    0.517   

In [15]:
# ============================================================
# STEP 9-B. 업종 분류 민감도 분석
# 반복성 55/60/65% × 효과크기 5/10/15%
# ============================================================

repeat_thresholds = [55, 60, 65]
effect_thresholds = [5, 10, 15]

sensitivity_results = []

for repeat_th in repeat_thresholds:
    for effect_th in effect_thresholds:

        temp = industry_month_evidence.copy()

        # 효과크기 기준
        upper = 1 + effect_th / 100
        lower = 1 - effect_th / 100

        # 월별 패턴 판정
        temp["상승패턴_temp"] = (
            (temp["이년이상_상승비율"] >= repeat_th)
            & (temp["삼년중앙효과"] >= upper)
        )

        temp["하락패턴_temp"] = (
            (temp["이년이상_하락비율"] >= repeat_th)
            & (temp["삼년중앙효과"] <= lower)
        )

        temp["패턴월_temp"] = (
            temp["상승패턴_temp"]
            | temp["하락패턴_temp"]
        )

        # 업종별 패턴 월 수
        temp_summary = (
            temp
            .groupby("업종_대분류")
            .agg(
                기업수=("기업수", "max"),
                패턴월수=("패턴월_temp", "sum")
            )
            .reset_index()
        )

        # 업종 전체 12개월 반복성 결합
        temp_summary = temp_summary.merge(
            industry_repeat[
                ["업종_대분류", "반복성_중앙값"]
            ],
            on="업종_대분류",
            how="left"
        )

        # 동일한 분류 규칙 적용
        def classify_sensitivity(row):

            if row["기업수"] < 50:
                return "판단보류"

            if (
                row["패턴월수"] >= 3
                and pd.notna(row["반복성_중앙값"])
                and row["반복성_중앙값"] >= 0.50
            ):
                return "① 다월"

            if 1 <= row["패턴월수"] <= 2:
                return "② 특정월"

            if row["패턴월수"] == 0:
                return "③ 비계절"

            return "판단보류"

        temp_summary["분류"] = temp_summary.apply(
            classify_sensitivity,
            axis=1
        )

        condition_name = f"R{repeat_th}_E{effect_th}"

        for _, row in temp_summary.iterrows():
            sensitivity_results.append({
                "업종_대분류": row["업종_대분류"],
                "조건": condition_name,
                "반복기준": repeat_th,
                "효과기준": effect_th,
                "패턴월수": row["패턴월수"],
                "분류": row["분류"]
            })


sensitivity_df = pd.DataFrame(sensitivity_results)


# ============================================================
# 1. 업종별 9개 조건 분류표
# ============================================================

classification_matrix = (
    sensitivity_df
    .pivot(
        index="업종_대분류",
        columns="조건",
        values="분류"
    )
)

# 보기 좋은 순서로 열 정렬
condition_order = [
    f"R{r}_E{e}"
    for r in repeat_thresholds
    for e in effect_thresholds
]

classification_matrix = classification_matrix[
    condition_order
]

print("=== 업종별 3×3 민감도 분류 ===")
print(classification_matrix.to_string())


# ============================================================
# 2. 업종별 분류 안정성
# ============================================================

def stability_summary(row):

    valid = row[row != "판단보류"]

    if len(valid) == 0:
        return pd.Series({
            "최빈분류": "판단보류",
            "최빈분류횟수": 0,
            "유효조건수": 0,
            "안정도_pct": np.nan
        })

    counts = valid.value_counts()

    return pd.Series({
        "최빈분류": counts.index[0],
        "최빈분류횟수": counts.iloc[0],
        "유효조건수": len(valid),
        "안정도_pct": counts.iloc[0] / len(valid) * 100
    })


stability = classification_matrix.apply(
    stability_summary,
    axis=1
)

# 기업수도 붙임
stability = (
    stability
    .merge(
        industry_size.rename("기업수"),
        left_index=True,
        right_index=True,
        how="left"
    )
)

print("\n=== 업종별 분류 안정성 ===")
print(
    stability[
        [
            "기업수",
            "최빈분류",
            "최빈분류횟수",
            "유효조건수",
            "안정도_pct"
        ]
    ]
    .sort_values(
        ["기업수"],
        ascending=False
    )
    .round(1)
    .to_string()
)


# ============================================================
# 3. 각 조건에서 다월/특정월/비계절 업종 수
# ============================================================

condition_counts = (
    sensitivity_df
    .groupby(["조건", "분류"])
    .size()
    .unstack(fill_value=0)
)

print("\n=== 기준별 업종 분류 개수 ===")
print(condition_counts)

=== 업종별 3×3 민감도 분류 ===
조건                       R55_E5 R55_E10 R55_E15 R60_E5 R60_E10 R60_E15 R65_E5 R65_E10 R65_E15
업종_대분류                                                                                       
건설업                        ① 다월   ② 특정월   ② 특정월   ① 다월   ② 특정월   ② 특정월  ② 특정월   ② 특정월   ② 특정월
광업                         판단보류    판단보류    판단보류   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류
교육 서비스업                    판단보류    판단보류    판단보류   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류
금융 및 보험업                   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류
농업, 임업 및 어업                판단보류    판단보류    판단보류   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류
도매 및 소매업                  ③ 비계절   ③ 비계절   ③ 비계절  ③ 비계절   ③ 비계절   ③ 비계절  ③ 비계절   ③ 비계절   ③ 비계절
보건업 및 사회복지 서비스업            판단보류    판단보류    판단보류   판단보류    판단보류    판단보류   판단보류    판단보류    판단보류
부동산업                      ③ 비계절   ③ 비계절   ③ 비계절  ③ 비계절   ③ 비계절   ③ 비계절  ③ 비계절   ③ 비계절   ③ 비계절
사업시설 관리, 사업 지원 및 임대 서비스업  ③ 비계절   ③ 비

In [16]:
# ============================================================
# STEP 10. 전체 가용 관측치를 이용한 업종별 계절성 통합 분석
#
# 핵심 원칙
# 1) 보간값 사용 안 함 → season_base 사용
# 2) 기업 전체 관측개월 수로 기업을 제거하지 않음
# 3) 기업×연도에 실제 관측이 6개월 이상이면 그 연도 사용
# 4) 기업×calendar month 기준:
#       3개년 비교 가능 → 가중치 2
#       2개년 비교 가능 → 가중치 1
#       1개년만 존재    → 반복성 분석 제외
# 5) 결측된 특정 월만 빠지고 다른 월은 그대로 활용
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. 원관측 데이터 준비
# ------------------------------------------------------------

all_base = season_base[
    [
        "법인ID",
        "기준월",
        "업종_대분류",
        "요구불입금금액"
    ]
].copy()

all_base["연도"] = all_base["기준월"].dt.year
all_base["월"] = all_base["기준월"].dt.month


print("전체 원관측 기업 수:",
      all_base["법인ID"].nunique())

print("전체 원관측 행 수:",
      len(all_base))


# ------------------------------------------------------------
# 1. 업종 정보가 기간 중 바뀌는 기업 확인
# ------------------------------------------------------------

industry_n = (
    all_base
    .dropna(subset=["업종_대분류"])
    .groupby("법인ID")["업종_대분류"]
    .nunique()
)

stable_ids = industry_n[
    industry_n == 1
].index

print("\n업종이 일관된 기업 수:",
      len(stable_ids))

print("업종이 2개 이상 나타나는 기업 수:",
      (industry_n > 1).sum())


# 업종 분류가 목적이므로 업종이 일관된 기업만 사용
all_base = all_base[
    all_base["법인ID"].isin(stable_ids)
    & all_base["업종_대분류"].notna()
].copy()


# ------------------------------------------------------------
# 2. 기업×연도별 실제 관측개월 수 계산
# ------------------------------------------------------------

firm_year_info = (
    all_base
    .groupby(["법인ID", "연도"])
    .agg(
        관측월수=("월", "nunique"),
        연간중앙값=("요구불입금금액", "median")
    )
    .reset_index()
)

print("\n기업×연도 관측개월 수 분포:")
print(
    firm_year_info["관측월수"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)


# ------------------------------------------------------------
# 3. 연간 대표수준 계산이 가능한 연도만 사용
#
# 최소 6개월 실제관측
# 연간 중앙값 > 0
# ------------------------------------------------------------

valid_firm_year = firm_year_info[
    (firm_year_info["관측월수"] >= 6)
    & (firm_year_info["연간중앙값"] > 0)
].copy()

valid_base = all_base.merge(
    valid_firm_year[
        [
            "법인ID",
            "연도",
            "관측월수",
            "연간중앙값"
        ]
    ],
    on=["법인ID", "연도"],
    how="inner"
)

valid_base["상대금액"] = (
    valid_base["요구불입금금액"]
    / valid_base["연간중앙값"]
)

print("\n사용 가능한 기업×연도 수:",
      len(valid_firm_year))

print("사용 가능한 기업 수:",
      valid_base["법인ID"].nunique())


# 기업당 사용 가능한 연도 수
usable_year_count = (
    valid_firm_year
    .groupby("법인ID")["연도"]
    .nunique()
)

print("\n기업별 사용 가능 연도 수:")
print(
    usable_year_count
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. 기업×calendar month별로
#    실제 몇 개 연도가 겹치는지 계산
# ------------------------------------------------------------

firm_month = (
    valid_base
    .groupby(
        ["법인ID", "업종_대분류", "월"]
    )
    .agg(
        비교가능연도수=("연도", "nunique"),

        상승연도수=(
            "상대금액",
            lambda x: (x > 1).sum()
        ),

        하락연도수=(
            "상대금액",
            lambda x: (x < 1).sum()
        ),

        기업월_중앙효과=(
            "상대금액",
            "median"
        )
    )
    .reset_index()
)


print("\n기업×월 비교가능 연도 수 전체 분포:")
print(
    firm_month["비교가능연도수"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 5. 최소 2개 연도에서 같은 달이 존재해야
#    '반복성' 평가 가능
# ------------------------------------------------------------

firm_month = firm_month[
    firm_month["비교가능연도수"] >= 2
].copy()


# 증거 가중치
#
# 2개년 = 1
# 3개년 = 2
#
# 3개년 자료가 2개년보다 두 배의 반복정보를 제공
firm_month["증거가중치"] = (
    firm_month["비교가능연도수"] - 1
)


# ------------------------------------------------------------
# 6. 반복 방향 정의
#
# 2개년:
#   두 해 모두 같은 방향이어야 반복 인정
#
# 3개년:
#   3년 중 2년 이상 같은 방향이면 반복 인정
# ------------------------------------------------------------

firm_month["반복상승"] = np.where(
    firm_month["비교가능연도수"] == 3,
    firm_month["상승연도수"] >= 2,
    firm_month["상승연도수"] == 2
)

firm_month["반복하락"] = np.where(
    firm_month["비교가능연도수"] == 3,
    firm_month["하락연도수"] >= 2,
    firm_month["하락연도수"] == 2
)


# 완전히 같은 방향도 진단용으로 보관
firm_month["완전상승"] = (
    firm_month["상승연도수"]
    == firm_month["비교가능연도수"]
)

firm_month["완전하락"] = (
    firm_month["하락연도수"]
    == firm_month["비교가능연도수"]
)


print("\n반복성 분석에 최종 사용되는 기업×월:")
print(len(firm_month))

print("\n그중 비교가능연도수:")
print(
    firm_month["비교가능연도수"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 7. 가중 중앙값 함수
#
# 3개년 기업의 효과크기를
# 2개년 기업보다 더 강하게 반영
# ------------------------------------------------------------

def weighted_median(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[mask]
    weights = weights[mask]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)

    cutoff = weights.sum() / 2

    return values[
        np.searchsorted(
            cumulative,
            cutoff,
            side="left"
        )
    ]


# ------------------------------------------------------------
# 8. 업종×월별 통합 증거 계산
# ------------------------------------------------------------

def summarize_month(g):

    w = g["증거가중치"].astype(float)

    total_weight = w.sum()

    return pd.Series({

        # 실제 참여 기업 수
        "비교기업수":
            len(g),

        "2개년기업수":
            (g["비교가능연도수"] == 2).sum(),

        "3개년기업수":
            (g["비교가능연도수"] == 3).sum(),

        # 전체 증거 중 3개년 자료 비중
        "3개년증거비중_pct":
            (
                w[
                    g["비교가능연도수"] == 3
                ].sum()
                / total_weight
                * 100
            ),

        # 반복 방향의 가중 비율
        "반복상승_가중비율":
            (
                (
                    w
                    * g["반복상승"].astype(int)
                ).sum()
                / total_weight
                * 100
            ),

        "반복하락_가중비율":
            (
                (
                    w
                    * g["반복하락"].astype(int)
                ).sum()
                / total_weight
                * 100
            ),

        # 더 강한 진단값
        "완전상승_가중비율":
            (
                (
                    w
                    * g["완전상승"].astype(int)
                ).sum()
                / total_weight
                * 100
            ),

        "완전하락_가중비율":
            (
                (
                    w
                    * g["완전하락"].astype(int)
                ).sum()
                / total_weight
                * 100
            ),

        # 효과 크기
        "가중중앙효과":
            weighted_median(
                g["기업월_중앙효과"],
                w
            )
    })


industry_month_all = (
    firm_month
    .groupby(
        ["업종_대분류", "월"],
        group_keys=False
    )
    .apply(
        summarize_month,
        include_groups=False
    )
    .reset_index()
)


# ------------------------------------------------------------
# 9. 기본 기준
#
# 반복성 ≥ 60%
# 효과 ≥ +10% 또는 ≤ -10%
# ------------------------------------------------------------

industry_month_all["상승패턴"] = (
    (
        industry_month_all[
            "반복상승_가중비율"
        ] >= 60
    )
    &
    (
        industry_month_all[
            "가중중앙효과"
        ] >= 1.10
    )
)

industry_month_all["하락패턴"] = (
    (
        industry_month_all[
            "반복하락_가중비율"
        ] >= 60
    )
    &
    (
        industry_month_all[
            "가중중앙효과"
        ] <= 0.90
    )
)

industry_month_all["패턴월"] = (
    industry_month_all["상승패턴"]
    | industry_month_all["하락패턴"]
)

industry_month_all["패턴방향"] = ""

industry_month_all.loc[
    industry_month_all["상승패턴"],
    "패턴방향"
] = "상승"

industry_month_all.loc[
    industry_month_all["하락패턴"],
    "패턴방향"
] = "하락"


# ------------------------------------------------------------
# 10. 업종별 요약
# ------------------------------------------------------------

industry_summary_all = (
    industry_month_all
    .groupby("업종_대분류")
    .agg(
        월별최소비교기업수=(
            "비교기업수",
            "min"
        ),
        월별최대비교기업수=(
            "비교기업수",
            "max"
        ),
        패턴월수=(
            "패턴월",
            "sum"
        ),
        평균_3개년증거비중=(
            "3개년증거비중_pct",
            "mean"
        )
    )
    .reset_index()
)


# 패턴 월 목록
pattern_list_all = (
    industry_month_all[
        industry_month_all["패턴월"]
    ]
    .assign(
        표시=lambda x:
            x["월"].astype(int).astype(str)
            + "월("
            + x["패턴방향"]
            + ", 반복="
            + x.apply(
                lambda r:
                    (
                        f"{r['반복상승_가중비율']:.1f}%"
                        if r["패턴방향"] == "상승"
                        else
                        f"{r['반복하락_가중비율']:.1f}%"
                    ),
                axis=1
            )
            + ", 효과="
            + x["가중중앙효과"]
              .round(2)
              .astype(str)
            + ")"
    )
    .groupby("업종_대분류")["표시"]
    .apply(", ".join)
    .rename("패턴월목록")
    .reset_index()
)

industry_summary_all = (
    industry_summary_all
    .merge(
        pattern_list_all,
        on="업종_대분류",
        how="left"
    )
)

industry_summary_all[
    "패턴월목록"
] = (
    industry_summary_all[
        "패턴월목록"
    ]
    .fillna("-")
)


# ------------------------------------------------------------
# 11. 기본 유형 분류
#
# 3개 이상 → 다월
# 1~2개 → 특정월
# 0개 → 비계절
#
# 단, 각 달 비교기업이 최소 50개 미만인 업종은 판단보류
# ------------------------------------------------------------

def classify_integrated(row):

    if row["월별최소비교기업수"] < 50:
        return "판단보류_표본부족"

    if row["패턴월수"] >= 3:
        return "① 다월 계절형"

    if row["패턴월수"] >= 1:
        return "② 특정월 계절형"

    return "③ 비계절형"


industry_summary_all["기본분류"] = (
    industry_summary_all
    .apply(
        classify_integrated,
        axis=1
    )
)


print("\n" + "=" * 100)
print("전체 가용관측 통합 기본 분류 - R60 / E10")
print("=" * 100)

print(
    industry_summary_all[
        [
            "업종_대분류",
            "월별최소비교기업수",
            "월별최대비교기업수",
            "평균_3개년증거비중",
            "패턴월수",
            "패턴월목록",
            "기본분류"
        ]
    ]
    .sort_values(
        [
            "기본분류",
            "월별최대비교기업수"
        ],
        ascending=[True, False]
    )
    .round(1)
    .to_string(index=False)
)


# ============================================================
# 12. 3 × 3 민감도 분석
# ============================================================

repeat_thresholds = [55, 60, 65]
effect_thresholds = [5, 10, 15]

sensitivity_rows = []

for repeat_th in repeat_thresholds:

    for effect_th in effect_thresholds:

        temp = industry_month_all.copy()

        upper = 1 + effect_th / 100
        lower = 1 - effect_th / 100

        temp["temp_pattern"] = (
            (
                (
                    temp[
                        "반복상승_가중비율"
                    ] >= repeat_th
                )
                &
                (
                    temp[
                        "가중중앙효과"
                    ] >= upper
                )
            )
            |
            (
                (
                    temp[
                        "반복하락_가중비율"
                    ] >= repeat_th
                )
                &
                (
                    temp[
                        "가중중앙효과"
                    ] <= lower
                )
            )
        )

        temp_sum = (
            temp
            .groupby("업종_대분류")
            .agg(
                월별최소비교기업수=(
                    "비교기업수",
                    "min"
                ),
                패턴월수=(
                    "temp_pattern",
                    "sum"
                )
            )
            .reset_index()
        )

        def temp_class(row):

            if row[
                "월별최소비교기업수"
            ] < 50:
                return "판단보류"

            if row["패턴월수"] >= 3:
                return "① 다월"

            if row["패턴월수"] >= 1:
                return "② 특정월"

            return "③ 비계절"

        temp_sum["분류"] = (
            temp_sum.apply(
                temp_class,
                axis=1
            )
        )

        condition = (
            f"R{repeat_th}_E{effect_th}"
        )

        for _, row in temp_sum.iterrows():

            sensitivity_rows.append({
                "업종_대분류":
                    row["업종_대분류"],

                "조건":
                    condition,

                "패턴월수":
                    row["패턴월수"],

                "분류":
                    row["분류"]
            })


sensitivity_all = pd.DataFrame(
    sensitivity_rows
)


condition_order = [
    f"R{r}_E{e}"
    for r in repeat_thresholds
    for e in effect_thresholds
]


classification_matrix_all = (
    sensitivity_all
    .pivot(
        index="업종_대분류",
        columns="조건",
        values="분류"
    )[condition_order]
)


# ------------------------------------------------------------
# 13. 분류 안정성
# ------------------------------------------------------------

def classification_stability(row):

    valid = row[
        row != "판단보류"
    ]

    if len(valid) == 0:

        return pd.Series({
            "최빈분류":
                "판단보류",

            "최빈횟수":
                0,

            "유효조건수":
                0,

            "안정도_pct":
                np.nan
        })

    counts = valid.value_counts()

    return pd.Series({
        "최빈분류":
            counts.index[0],

        "최빈횟수":
            counts.iloc[0],

        "유효조건수":
            len(valid),

        "안정도_pct":
            counts.iloc[0]
            / len(valid)
            * 100
    })


stability_all = (
    classification_matrix_all
    .apply(
        classification_stability,
        axis=1
    )
)


print("\n" + "=" * 100)
print("전체 가용관측 3×3 민감도 분류")
print("=" * 100)

print(
    classification_matrix_all
    .to_string()
)


print("\n" + "=" * 100)
print("전체 가용관측 분류 안정성")
print("=" * 100)

print(
    stability_all
    .sort_values(
        "안정도_pct",
        ascending=False
    )
    .round(1)
    .to_string()
)

전체 원관측 기업 수: 15473
전체 원관측 행 수: 365988

업종이 일관된 기업 수: 15473
업종이 2개 이상 나타나는 기업 수: 0

기업×연도 관측개월 수 분포:
count    36858.000000
mean         9.929676
std          3.184830
min          1.000000
10%          4.000000
25%          9.000000
50%         12.000000
75%         12.000000
90%         12.000000
max         12.000000
Name: 관측월수, dtype: float64

사용 가능한 기업×연도 수: 29078
사용 가능한 기업 수: 12734

기업별 사용 가능 연도 수:
연도
1    3187
2    2750
3    6797
Name: count, dtype: int64

기업×월 비교가능 연도 수 전체 분포:
비교가능연도수
1    39197
2    34329
3    71897
Name: count, dtype: int64

반복성 분석에 최종 사용되는 기업×월:
106226

그중 비교가능연도수:
비교가능연도수
2    34329
3    71897
Name: count, dtype: int64

전체 가용관측 통합 기본 분류 - R60 / E10
                  업종_대분류  월별최소비교기업수  월별최대비교기업수  평균_3개년증거비중  패턴월수                                                                                                      패턴월목록      기본분류
                     건설업     1287.0     1361.0        84.7     1                                                                      

In [17]:
# ============================================================
# STEP 11-A. 표본구성 민감도
# 기업×연도 최소 관측월 = 5 / 6 / 7 / 8
#
# 계절성 판정기준은 고정:
# 반복성 >= 60%
# 효과크기 >= +10% 또는 <= -10%
#
# 2개년 증거 가중치 = 1
# 3개년 증거 가중치 = 2
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 가중 중앙값
# ------------------------------------------------------------

def weighted_median(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[mask]
    weights = weights[mask]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)
    cutoff = weights.sum() / 2

    return values[
        np.searchsorted(
            cumulative,
            cutoff,
            side="left"
        )
    ]


# ------------------------------------------------------------
# 최소 관측월 조건 하나를 넣으면
# 업종×월 계절성 근거를 계산하는 함수
# ------------------------------------------------------------

def build_season_evidence(min_months=6, industry_map=None):

    base = season_base[
        [
            "법인ID",
            "기준월",
            "업종_대분류",
            "요구불입금금액"
        ]
    ].copy()

    base["연도"] = base["기준월"].dt.year
    base["월"] = base["기준월"].dt.month


    # 업종이 기간 중 변하지 않는 기업만
    industry_n = (
        base
        .dropna(subset=["업종_대분류"])
        .groupby("법인ID")["업종_대분류"]
        .nunique()
    )

    stable_ids = industry_n[
        industry_n == 1
    ].index

    base = base[
        base["법인ID"].isin(stable_ids)
        & base["업종_대분류"].notna()
    ].copy()


    # 필요하면 일부 업종을 '기타업종'으로 변환
    if industry_map is None:

        base["분석업종"] = (
            base["업종_대분류"]
        )

    else:

        base["분석업종"] = (
            base["업종_대분류"]
            .replace(industry_map)
        )


    # 기업×연도 수준
    firm_year = (
        base
        .groupby(["법인ID", "연도"])
        .agg(
            관측월수=("월", "nunique"),
            연간중앙값=("요구불입금금액", "median")
        )
        .reset_index()
    )


    valid_firm_year = firm_year[
        (firm_year["관측월수"] >= min_months)
        & (firm_year["연간중앙값"] > 0)
    ].copy()


    valid = base.merge(
        valid_firm_year[
            [
                "법인ID",
                "연도",
                "연간중앙값"
            ]
        ],
        on=["법인ID", "연도"],
        how="inner"
    )


    valid["상대금액"] = (
        valid["요구불입금금액"]
        / valid["연간중앙값"]
    )


    # 기업×calendar month
    firm_month = (
        valid
        .groupby(
            [
                "법인ID",
                "분석업종",
                "월"
            ]
        )
        .agg(
            비교가능연도수=(
                "연도",
                "nunique"
            ),

            상승연도수=(
                "상대금액",
                lambda x: (x > 1).sum()
            ),

            하락연도수=(
                "상대금액",
                lambda x: (x < 1).sum()
            ),

            기업월_중앙효과=(
                "상대금액",
                "median"
            )
        )
        .reset_index()
    )


    # 반복을 확인하려면 2개년 이상 필요
    firm_month = firm_month[
        firm_month["비교가능연도수"] >= 2
    ].copy()


    # 2개년 = 1
    # 3개년 = 2
    firm_month["증거가중치"] = (
        firm_month["비교가능연도수"] - 1
    )


    # 2개년이면 두 해 모두 같은 방향
    # 3개년이면 2년 이상 같은 방향
    firm_month["반복상승"] = np.where(
        firm_month["비교가능연도수"] == 3,
        firm_month["상승연도수"] >= 2,
        firm_month["상승연도수"] == 2
    )

    firm_month["반복하락"] = np.where(
        firm_month["비교가능연도수"] == 3,
        firm_month["하락연도수"] >= 2,
        firm_month["하락연도수"] == 2
    )


    # 업종×월 집계
    rows = []

    for (industry, month), g in firm_month.groupby(
        ["분석업종", "월"]
    ):

        w = g["증거가중치"].astype(float)

        total_w = w.sum()

        rows.append({

            "업종_대분류": industry,

            "월": month,

            "비교기업수": len(g),

            "2개년기업수":
                (g["비교가능연도수"] == 2).sum(),

            "3개년기업수":
                (g["비교가능연도수"] == 3).sum(),

            "반복상승_가중비율":
                (
                    (
                        w
                        * g["반복상승"].astype(int)
                    ).sum()
                    / total_w
                    * 100
                ),

            "반복하락_가중비율":
                (
                    (
                        w
                        * g["반복하락"].astype(int)
                    ).sum()
                    / total_w
                    * 100
                ),

            "가중중앙효과":
                weighted_median(
                    g["기업월_중앙효과"],
                    w
                )
        })


    evidence = pd.DataFrame(rows)


    # --------------------------------------------------------
    # R60 / E10
    # --------------------------------------------------------

    evidence["상승패턴"] = (
        (
            evidence["반복상승_가중비율"]
            >= 60
        )
        &
        (
            evidence["가중중앙효과"]
            >= 1.10
        )
    )

    evidence["하락패턴"] = (
        (
            evidence["반복하락_가중비율"]
            >= 60
        )
        &
        (
            evidence["가중중앙효과"]
            <= 0.90
        )
    )

    evidence["패턴월"] = (
        evidence["상승패턴"]
        | evidence["하락패턴"]
    )

    evidence["방향"] = ""

    evidence.loc[
        evidence["상승패턴"],
        "방향"
    ] = "상승"

    evidence.loc[
        evidence["하락패턴"],
        "방향"
    ] = "하락"


    # 업종별 요약
    summary = (
        evidence
        .groupby("업종_대분류")
        .agg(
            월수=("월", "nunique"),
            월별최소비교기업수=("비교기업수", "min"),
            월별최대비교기업수=("비교기업수", "max"),
            패턴월수=("패턴월", "sum")
        )
        .reset_index()
    )


    # 패턴 월 문자열
    pattern_rows = evidence[
        evidence["패턴월"]
    ].copy()

    if len(pattern_rows) > 0:

        pattern_rows["표시"] = (
            pattern_rows["월"].astype(int).astype(str)
            + "월("
            + pattern_rows["방향"]
            + ")"
        )

        pattern_list = (
            pattern_rows
            .groupby("업종_대분류")["표시"]
            .apply(", ".join)
            .rename("패턴월목록")
            .reset_index()
        )

        summary = summary.merge(
            pattern_list,
            on="업종_대분류",
            how="left"
        )

    else:

        summary["패턴월목록"] = "-"


    summary["패턴월목록"] = (
        summary["패턴월목록"]
        .fillna("-")
    )


    def classify(row):

        if (
            row["월수"] < 12
            or row["월별최소비교기업수"] < 50
        ):
            return "판단보류_표본부족"

        if row["패턴월수"] >= 3:
            return "① 다월 계절형"

        if row["패턴월수"] >= 1:
            return "② 특정월 계절형"

        return "③ 비계절형"


    summary["분류"] = summary.apply(
        classify,
        axis=1
    )

    diagnostics = {
        "최소관측월": min_months,
        "사용기업수": valid["법인ID"].nunique(),
        "사용기업연도수": len(valid_firm_year),
        "반복분석_기업월수": len(firm_month)
    }

    return evidence, summary, diagnostics


# ============================================================
# 5 / 6 / 7 / 8개월 조건 비교
# ============================================================

min_month_list = [5, 6, 7, 8]

sample_sensitivity = []
diagnostics_list = []

for m in min_month_list:

    ev, sm, diag = build_season_evidence(
        min_months=m
    )

    diagnostics_list.append(diag)

    for _, row in sm.iterrows():

        sample_sensitivity.append({

            "업종_대분류":
                row["업종_대분류"],

            "최소관측월":
                m,

            "패턴월수":
                row["패턴월수"],

            "패턴월목록":
                row["패턴월목록"],

            "분류":
                row["분류"]
        })


sample_sensitivity = pd.DataFrame(
    sample_sensitivity
)

diagnostics_df = pd.DataFrame(
    diagnostics_list
)


print("=" * 90)
print("최소 관측월별 사용 데이터 규모")
print("=" * 90)
print(
    diagnostics_df.to_string(index=False)
)


# 분류 매트릭스
sample_class_matrix = (
    sample_sensitivity
    .pivot(
        index="업종_대분류",
        columns="최소관측월",
        values="분류"
    )
)

print("\n" + "=" * 90)
print("5/6/7/8개월 표본구성 민감도 - 업종 분류")
print("=" * 90)

print(
    sample_class_matrix.to_string()
)


# 패턴월 매트릭스
sample_pattern_matrix = (
    sample_sensitivity
    .pivot(
        index="업종_대분류",
        columns="최소관측월",
        values="패턴월목록"
    )
)

print("\n" + "=" * 90)
print("5/6/7/8개월 표본구성 민감도 - 패턴월")
print("=" * 90)

print(
    sample_pattern_matrix.to_string()
)


# ------------------------------------------------------------
# 분류 안정성
# ------------------------------------------------------------

def sample_stability(row):

    valid = row[
        ~row.astype(str).str.startswith("판단보류")
    ]

    if len(valid) == 0:

        return pd.Series({
            "최빈분류": "판단보류",
            "최빈횟수": 0,
            "유효조건수": 0,
            "표본구성안정도_pct": np.nan
        })

    counts = valid.value_counts()

    return pd.Series({
        "최빈분류": counts.index[0],
        "최빈횟수": counts.iloc[0],
        "유효조건수": len(valid),
        "표본구성안정도_pct":
            counts.iloc[0] / len(valid) * 100
    })


sample_stability_table = (
    sample_class_matrix
    .apply(
        sample_stability,
        axis=1
    )
)

print("\n" + "=" * 90)
print("표본구성 분류 안정성")
print("=" * 90)

print(
    sample_stability_table
    .sort_values(
        "표본구성안정도_pct",
        ascending=False
    )
    .round(1)
    .to_string()
)

최소 관측월별 사용 데이터 규모
 최소관측월  사용기업수  사용기업연도수  반복분석_기업월수
     5  13006    29888     107698
     6  12734    29078     106226
     7  12411    28189     104226
     8  12101    27378     102236

5/6/7/8개월 표본구성 민감도 - 업종 분류
최소관측월                             5          6          7          8
업종_대분류                                                              
건설업                       ② 특정월 계절형  ② 특정월 계절형  ② 특정월 계절형  ② 특정월 계절형
광업                        판단보류_표본부족  판단보류_표본부족  판단보류_표본부족  판단보류_표본부족
교육 서비스업                      ③ 비계절형     ③ 비계절형     ③ 비계절형     ③ 비계절형
금융 및 보험업                     ③ 비계절형     ③ 비계절형     ③ 비계절형     ③ 비계절형
농업, 임업 및 어업               판단보류_표본부족  판단보류_표본부족  판단보류_표본부족  판단보류_표본부족
도매 및 소매업                     ③ 비계절형     ③ 비계절형     ③ 비계절형     ③ 비계절형
보건업 및 사회복지 서비스업           판단보류_표본부족  판단보류_표본부족  판단보류_표본부족  판단보류_표본부족
부동산업                         ③ 비계절형     ③ 비계절형     ③ 비계절형     ③ 비계절형
사업시설 관리, 사업 지원 및 임대 서비스업     ③ 비계절형     ③ 비계절형     ③ 비계절형     ③ 비계절형
수도, 하수 및 폐기물 처리, 원료 재생업  

In [18]:
# ============================================================
# STEP 11-B. 표본부족 업종 → 기타업종 통합 검정
# ============================================================

# 먼저 최소 6개월 기준 결과
baseline_evidence, baseline_summary, _ = (
    build_season_evidence(
        min_months=6
    )
)

# 표본부족 업종 확인
small_industries = (
    baseline_summary.loc[
        baseline_summary["분류"]
        == "판단보류_표본부족",
        "업종_대분류"
    ]
    .tolist()
)

print("기타업종으로 묶을 업종:")
for x in small_industries:
    print("-", x)


# 업종명 매핑
other_map = {
    x: "기타업종"
    for x in small_industries
}


# 동일 방법으로 재계산
other_evidence, other_summary, other_diag = (
    build_season_evidence(
        min_months=6,
        industry_map=other_map
    )
)


# 기타업종 월별 결과
other_month = other_evidence[
    other_evidence["업종_대분류"]
    == "기타업종"
].copy()


print("\n" + "=" * 90)
print("기타업종 월별 계절성 근거")
print("=" * 90)

print(
    other_month[
        [
            "월",
            "비교기업수",
            "2개년기업수",
            "3개년기업수",
            "반복상승_가중비율",
            "반복하락_가중비율",
            "가중중앙효과",
            "패턴월",
            "방향"
        ]
    ]
    .round(3)
    .to_string(index=False)
)


print("\n" + "=" * 90)
print("기타업종 기본 분류 R60 / E10")
print("=" * 90)

print(
    other_summary[
        other_summary["업종_대분류"]
        == "기타업종"
    ].to_string(index=False)
)


# ============================================================
# 기타업종에 대해서도 3×3 민감도 확인
# ============================================================

other_sensitivity = []

for r_th in [55, 60, 65]:

    for e_th in [5, 10, 15]:

        temp = other_month.copy()

        upper = 1 + e_th / 100
        lower = 1 - e_th / 100

        temp["패턴"] = (
            (
                (temp["반복상승_가중비율"] >= r_th)
                & (temp["가중중앙효과"] >= upper)
            )
            |
            (
                (temp["반복하락_가중비율"] >= r_th)
                & (temp["가중중앙효과"] <= lower)
            )
        )

        pattern_count = int(
            temp["패턴"].sum()
        )

        if (
            len(temp) < 12
            or temp["비교기업수"].min() < 50
        ):
            result = "판단보류"

        elif pattern_count >= 3:
            result = "① 다월"

        elif pattern_count >= 1:
            result = "② 특정월"

        else:
            result = "③ 비계절"

        pattern_months = (
            temp.loc[
                temp["패턴"],
                "월"
            ]
            .astype(int)
            .tolist()
        )

        other_sensitivity.append({
            "조건": f"R{r_th}_E{e_th}",
            "패턴월수": pattern_count,
            "패턴월": pattern_months,
            "분류": result
        })


other_sensitivity = pd.DataFrame(
    other_sensitivity
)

print("\n" + "=" * 90)
print("기타업종 3×3 민감도")
print("=" * 90)

print(
    other_sensitivity
    .to_string(index=False)
)

기타업종으로 묶을 업종:
- 광업
- 농업, 임업 및 어업
- 보건업 및 사회복지 서비스업
- 전기, 가스, 증기 및 공기조절 공급업

기타업종 월별 계절성 근거
 월  비교기업수  2개년기업수  3개년기업수  반복상승_가중비율  반복하락_가중비율  가중중앙효과   패턴월 방향
 1    141      46      95     32.203     46.610   0.984 False   
 2    133      39      94     35.242     44.493   0.987 False   
 3    139      49      90     31.878     47.162   0.988 False   
 4    141      41     100     46.888     33.195   1.022 False   
 5    142      46      96     47.479     29.412   1.027 False   
 6    152      55      97     37.349     31.325   1.000 False   
 7    149      48     101     48.400     25.600   1.024 False   
 8    153      50     103     33.203     33.203   1.000 False   
 9    138      39      99     36.287     33.333   1.000 False   
10    139      41      98     37.553     38.397   1.000 False   
11    146      49      97     25.103     51.029   0.960 False   
12    135      40      95     37.826     40.435   1.000 False   

기타업종 기본 분류 R60 / E10
업종_대분류  월수  월별최소비교기업수  월별최대비교기업수  패턴월수 패턴월

In [19]:
# ============================================================
# STEP 12. 특정월 계절형 업종의 12월 조정지수 후보 산정
#
# 대상
# 1) 건설업
# 2) 전문, 과학 및 기술 서비스업
#
# 원칙
# - 보간값 사용 안 함
# - 기업×연도별 12월 제외 baseline 사용
# - baseline 계산에 비12월 실제관측 최소 6개월 필요
# - 기업 규모 제거
# - 중앙값 중심 robust 산정
# - Leave-One-Year-Out 검증
# ============================================================

import pandas as pd
import numpy as np


TARGET_INDUSTRIES = [
    "건설업",
    "전문, 과학 및 기술 서비스업"
]


# ------------------------------------------------------------
# 1. 대상 데이터
# ------------------------------------------------------------

idx_base = season_base.loc[
    season_base["업종_대분류"].isin(TARGET_INDUSTRIES),
    [
        "법인ID",
        "기준월",
        "업종_대분류",
        "요구불입금금액"
    ]
].copy()

idx_base["연도"] = idx_base["기준월"].dt.year
idx_base["월"] = idx_base["기준월"].dt.month


# ------------------------------------------------------------
# 2. 기업×연도별 '12월 제외 baseline' 계산
#
# 12월 자체가 baseline을 끌어올리는 것을 방지
# ------------------------------------------------------------

non_dec = idx_base[
    idx_base["월"] != 12
].copy()

firm_year_baseline = (
    non_dec
    .groupby(
        ["법인ID", "업종_대분류", "연도"]
    )
    .agg(
        비12월관측수=("월", "nunique"),
        비12월중앙값=("요구불입금금액", "median")
    )
    .reset_index()
)


# baseline 신뢰성을 위해
# 12월 제외 실제관측이 최소 6개월인 기업×연도 사용
firm_year_baseline = firm_year_baseline[
    (firm_year_baseline["비12월관측수"] >= 6)
    & (firm_year_baseline["비12월중앙값"] > 0)
].copy()


# ------------------------------------------------------------
# 3. 실제 12월 관측값 연결
# ------------------------------------------------------------

dec = idx_base[
    idx_base["월"] == 12
][
    [
        "법인ID",
        "업종_대분류",
        "연도",
        "요구불입금금액"
    ]
].copy()

dec = dec.rename(
    columns={
        "요구불입금금액": "12월금액"
    }
)


dec_ratio = firm_year_baseline.merge(
    dec,
    on=[
        "법인ID",
        "업종_대분류",
        "연도"
    ],
    how="inner"
)


dec_ratio["12월효과"] = (
    dec_ratio["12월금액"]
    / dec_ratio["비12월중앙값"]
)


print("지수 산정용 기업×연도 수:")
print(
    dec_ratio
    .groupby(["업종_대분류", "연도"])
    .size()
    .unstack(fill_value=0)
)


# ------------------------------------------------------------
# 4. 연도별 robust 12월 효과
# ------------------------------------------------------------

year_effect = (
    dec_ratio
    .groupby(
        ["업종_대분류", "연도"]
    )["12월효과"]
    .agg(
        기업수="count",
        중앙효과="median",
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75)
    )
    .reset_index()
)


print("\n연도별 12월 효과:")
print(
    year_effect
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. 후보 A
# 연도별 중앙효과를 다시 중앙값
# → 각 연도 동일 가중
# ------------------------------------------------------------

candidate_A = (
    year_effect
    .groupby("업종_대분류")["중앙효과"]
    .median()
    .rename("후보A_연도균등지수")
)


# ------------------------------------------------------------
# 6. 후보 B
# 모든 기업×연도 12월 효과 pooled median
# ------------------------------------------------------------

candidate_B = (
    dec_ratio
    .groupby("업종_대분류")["12월효과"]
    .median()
    .rename("후보B_통합중앙지수")
)


candidate_index = pd.concat(
    [candidate_A, candidate_B],
    axis=1
)

candidate_index["A_B차이_pct"] = (
    (
        candidate_index["후보A_연도균등지수"]
        / candidate_index["후보B_통합중앙지수"]
        - 1
    )
    * 100
)


print("\n후보 계절지수:")
print(
    candidate_index
    .round(3)
    .to_string()
)


# ============================================================
# 7. Leave-One-Year-Out 검증
#
# 해당 연도를 제외한 나머지 2개 연도의
# 연도별 중앙효과 중앙값을 지수로 사용
# ============================================================

loo_rows = []

years = [2023, 2024, 2025]

for industry in TARGET_INDUSTRIES:

    ind_year = year_effect[
        year_effect["업종_대분류"] == industry
    ].copy()

    ind_dec = dec_ratio[
        dec_ratio["업종_대분류"] == industry
    ].copy()

    for test_year in years:

        train_years = [
            y for y in years
            if y != test_year
        ]

        train_effects = (
            ind_year.loc[
                ind_year["연도"].isin(train_years),
                "중앙효과"
            ]
        )

        if len(train_effects) == 0:
            continue

        # 다른 연도들만으로 만든 지수
        train_index = train_effects.median()

        test = ind_dec[
            ind_dec["연도"] == test_year
        ].copy()

        if len(test) == 0:
            continue

        # 조정 후 12월 효과
        test["조정후효과"] = (
            test["12월효과"]
            / train_index
        )

        loo_rows.append({

            "업종_대분류": industry,

            "검증연도": test_year,

            "학습연도":
                ",".join(
                    map(str, train_years)
                ),

            "사용지수":
                train_index,

            "검증기업수":
                len(test),

            "조정전_중앙효과":
                test["12월효과"].median(),

            "조정후_중앙효과":
                test["조정후효과"].median(),

            "조정후_Q25":
                test["조정후효과"].quantile(0.25),

            "조정후_Q75":
                test["조정후효과"].quantile(0.75),

            # 조정 후에도 10% 이상 높은 기업 비율
            "조정후_10퍼초과비율":
                (
                    (test["조정후효과"] > 1.10)
                    .mean()
                    * 100
                ),

            # 조정 후 0.9~1.1 안에 들어온 기업 비율
            "조정후_±10퍼내비율":
                (
                    test["조정후효과"]
                    .between(0.90, 1.10)
                    .mean()
                    * 100
                )
        })


loo_result = pd.DataFrame(
    loo_rows
)


print("\n" + "=" * 100)
print("Leave-One-Year-Out 검증")
print("=" * 100)

print(
    loo_result
    .round(3)
    .to_string(index=False)
)

지수 산정용 기업×연도 수:
연도                2023  2024  2025
업종_대분류                            
건설업               1272  1272  1255
전문, 과학 및 기술 서비스업   225   245   254

연도별 12월 효과:
          업종_대분류   연도  기업수  중앙효과   Q25   Q75
             건설업 2023 1272 1.652 0.932 3.595
             건설업 2024 1272 1.667 0.909 3.598
             건설업 2025 1255 1.857 0.944 3.714
전문, 과학 및 기술 서비스업 2023  225 1.188 0.846 2.577
전문, 과학 및 기술 서비스업 2024  245 1.375 0.925 2.609
전문, 과학 및 기술 서비스업 2025  254 1.178 0.920 2.339

후보 계절지수:
                  후보A_연도균등지수  후보B_통합중앙지수  A_B차이_pct
업종_대분류                                             
건설업                    1.667       1.727     -3.509
전문, 과학 및 기술 서비스업       1.188       1.253     -5.148

Leave-One-Year-Out 검증
          업종_대분류  검증연도      학습연도  사용지수  검증기업수  조정전_중앙효과  조정후_중앙효과  조정후_Q25  조정후_Q75  조정후_10퍼초과비율  조정후_±10퍼내비율
             건설업  2023 2024,2025 1.762   1272     1.652     0.938    0.529    2.040       43.711        8.097
             건설업  2024 2023,2025 1.755   1272     1.667

In [20]:
# ============================================================
# STEP 12-A. 팀 방식 vs 우리 방식
# 동일 원관측 표본에서 월별 계절효과 직접 비교
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. 동일한 원관측 표본
# ------------------------------------------------------------

compare_base = season_base[
    [
        "법인ID",
        "기준월",
        "업종_대분류",
        "요구불입금금액"
    ]
].copy()

compare_base["연도"] = compare_base["기준월"].dt.year
compare_base["월"] = compare_base["기준월"].dt.month


# log1p 사용 전 음수 확인
negative_count = (
    compare_base["요구불입금금액"] < 0
).sum()

print("음수 금액 행 수:", negative_count)

if negative_count > 0:
    raise ValueError(
        "요구불입금금액에 음수가 있어 log1p 방식을 바로 사용할 수 없습니다."
    )


# ============================================================
# PART A. 팀 방식
#
# 1) log1p
# 2) 기업의 전체 관측기간 평균 제거
# 3) 연도 전체 중앙효과 제거
# 4) 연도×월 중앙값
# ============================================================

team = compare_base.copy()

team["log_입금"] = np.log1p(
    team["요구불입금금액"]
)

team["기업전체평균_log"] = (
    team
    .groupby("법인ID")["log_입금"]
    .transform("mean")
)

team["기업평소대비"] = (
    team["log_입금"]
    - team["기업전체평균_log"]
)

# 연도 자체의 전체적인 상승/하락 제거
team["연도조정값"] = (
    team["기업평소대비"]
    - team
      .groupby("연도")["기업평소대비"]
      .transform("median")
)


# 연도×월 효과
team_year_month = (
    team
    .groupby(["연도", "월"])["연도조정값"]
    .median()
    .reset_index()
)

team_year_month["효과_pct"] = (
    np.exp(team_year_month["연도조정값"]) - 1
) * 100


team_pivot = (
    team_year_month
    .pivot(
        index="월",
        columns="연도",
        values="효과_pct"
    )
)


# 3개 연도를 동일하게 반영한 대표 효과
team_equal_year = (
    team_year_month
    .groupby("월")["연도조정값"]
    .median()
)

team_equal_year_pct = (
    np.exp(team_equal_year) - 1
) * 100


# ============================================================
# PART B. 우리 방식
#
# 기업×연도 내부의 평소 수준 제거
# 동일 기업·동일 월을 2~3개년 반복 비교
#
# 기업×연도 실제관측 6개월 이상
# 2개년 증거=1
# 3개년 증거=2
# ============================================================

our = compare_base.copy()


# 기업×연도 대표 수준
firm_year = (
    our
    .groupby(["법인ID", "연도"])
    .agg(
        관측월수=("월", "nunique"),
        연간중앙값=("요구불입금금액", "median")
    )
    .reset_index()
)


valid_firm_year = firm_year[
    (firm_year["관측월수"] >= 6)
    & (firm_year["연간중앙값"] > 0)
].copy()


our = our.merge(
    valid_firm_year[
        [
            "법인ID",
            "연도",
            "연간중앙값"
        ]
    ],
    on=["법인ID", "연도"],
    how="inner"
)


our["상대금액"] = (
    our["요구불입금금액"]
    / our["연간중앙값"]
)


# 기업×calendar month
our_firm_month = (
    our
    .groupby(["법인ID", "월"])
    .agg(
        비교가능연도수=("연도", "nunique"),

        상승연도수=(
            "상대금액",
            lambda x: (x > 1).sum()
        ),

        하락연도수=(
            "상대금액",
            lambda x: (x < 1).sum()
        ),

        기업월_중앙효과=(
            "상대금액",
            "median"
        )
    )
    .reset_index()
)


# 최소 2개년 비교
our_firm_month = our_firm_month[
    our_firm_month["비교가능연도수"] >= 2
].copy()


# 3개년 = 가중치 2
# 2개년 = 가중치 1
our_firm_month["증거가중치"] = (
    our_firm_month["비교가능연도수"] - 1
)


# 반복방향
our_firm_month["반복상승"] = np.where(
    our_firm_month["비교가능연도수"] == 3,
    our_firm_month["상승연도수"] >= 2,
    our_firm_month["상승연도수"] == 2
)

our_firm_month["반복하락"] = np.where(
    our_firm_month["비교가능연도수"] == 3,
    our_firm_month["하락연도수"] >= 2,
    our_firm_month["하락연도수"] == 2
)


# ------------------------------------------------------------
# 가중 중앙값
# ------------------------------------------------------------

def weighted_median(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[mask]
    weights = weights[mask]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)

    return values[
        np.searchsorted(
            cumulative,
            weights.sum() / 2,
            side="left"
        )
    ]


# ------------------------------------------------------------
# 전체 기업 기준 월별 증거
# ------------------------------------------------------------

our_rows = []

for month, g in our_firm_month.groupby("월"):

    w = g["증거가중치"].astype(float)

    total_w = w.sum()

    effect = weighted_median(
        g["기업월_중앙효과"],
        w
    )

    our_rows.append({

        "월": month,

        "비교기업수":
            len(g),

        "2개년기업수":
            (g["비교가능연도수"] == 2).sum(),

        "3개년기업수":
            (g["비교가능연도수"] == 3).sum(),

        "반복상승_pct":
            (
                (
                    w
                    * g["반복상승"].astype(int)
                ).sum()
                / total_w
                * 100
            ),

        "반복하락_pct":
            (
                (
                    w
                    * g["반복하락"].astype(int)
                ).sum()
                / total_w
                * 100
            ),

        "우리_중앙효과":
            effect,

        "우리_효과_pct":
            (effect - 1) * 100
    })


our_month = pd.DataFrame(
    our_rows
).set_index("월")


# ============================================================
# PART C. 두 방법 결합
# ============================================================

comparison = pd.DataFrame(
    index=range(1, 13)
)

comparison.index.name = "월"


for year in [2023, 2024, 2025]:

    comparison[
        f"팀_{year}_pct"
    ] = team_pivot[year]


comparison[
    "팀_3년대표효과_pct"
] = team_equal_year_pct


comparison = comparison.join(
    our_month[
        [
            "비교기업수",
            "2개년기업수",
            "3개년기업수",
            "반복상승_pct",
            "반복하락_pct",
            "우리_효과_pct"
        ]
    ]
)


# ------------------------------------------------------------
# 팀 방식의 연도별 방향 반복 여부
# ------------------------------------------------------------

def team_direction(row):

    vals = np.array([
        row["팀_2023_pct"],
        row["팀_2024_pct"],
        row["팀_2025_pct"]
    ])

    if np.all(vals > 0):
        return "3년 모두 상승"

    if np.all(vals < 0):
        return "3년 모두 하락"

    return "부호 불일치"


comparison["팀_연도방향"] = (
    comparison.apply(
        team_direction,
        axis=1
    )
)


# ------------------------------------------------------------
# 우리가 기존에 사용한 R60 / E10 기준도 표시
# ------------------------------------------------------------

def our_strong_pattern(row):

    if (
        row["반복상승_pct"] >= 60
        and row["우리_효과_pct"] >= 10
    ):
        return "강한 상승패턴"

    if (
        row["반복하락_pct"] >= 60
        and row["우리_효과_pct"] <= -10
    ):
        return "강한 하락패턴"

    return "강한 패턴 아님"


comparison["우리_R60_E10"] = (
    comparison.apply(
        our_strong_pattern,
        axis=1
    )
)


# ============================================================
# 핵심 월만 출력
# ============================================================

focus_months = [2, 8, 10, 11, 12]

print("\n" + "=" * 120)
print("팀 방식 vs 우리 방식 - 핵심 월")
print("=" * 120)

print(
    comparison.loc[
        focus_months,
        [
            "팀_2023_pct",
            "팀_2024_pct",
            "팀_2025_pct",
            "팀_3년대표효과_pct",
            "팀_연도방향",
            "비교기업수",
            "반복상승_pct",
            "반복하락_pct",
            "우리_효과_pct",
            "우리_R60_E10"
        ]
    ]
    .round(2)
    .to_string()
)


print("\n" + "=" * 120)
print("참고: 1~12월 전체 비교")
print("=" * 120)

print(
    comparison[
        [
            "팀_3년대표효과_pct",
            "팀_연도방향",
            "반복상승_pct",
            "반복하락_pct",
            "우리_효과_pct",
            "우리_R60_E10"
        ]
    ]
    .round(2)
    .to_string()
)

음수 금액 행 수: 0

팀 방식 vs 우리 방식 - 핵심 월
    팀_2023_pct  팀_2024_pct  팀_2025_pct  팀_3년대표효과_pct    팀_연도방향  비교기업수  반복상승_pct  반복하락_pct  우리_효과_pct 우리_R60_E10
월                                                                                                              
2        -2.91       -1.32       -4.14         -2.91  3년 모두 하락   8619     34.20     44.47       0.00   강한 패턴 아님
8        -0.91       -4.61       -6.18         -4.61  3년 모두 하락   9157     30.90     46.06      -0.99   강한 패턴 아님
10       -0.42       -1.71       -0.53         -0.53  3년 모두 하락   8929     36.04     41.39       0.00   강한 패턴 아님
11        0.55       -4.87       -7.00         -4.87    부호 불일치   8803     31.71     46.40      -1.33   강한 패턴 아님
12        7.19        8.90       12.35          8.90  3년 모두 상승   8556     53.10     27.86       7.43   강한 패턴 아님

참고: 1~12월 전체 비교
    팀_3년대표효과_pct    팀_연도방향  반복상승_pct  반복하락_pct  우리_효과_pct 우리_R60_E10
월                                                                   
1          -1.01    부호 불일치

In [21]:
# ============================================================
# STEP 13. 전체 기업 공통 계절지수 후보
#
# 기업×연도 효과 제거 → calendar month 효과 추정
# 보간값 사용 안 함
# 기업×연도 실제관측 최소 6개월
# ============================================================

import pandas as pd
import numpy as np


base = season_base[
    [
        "법인ID",
        "기준월",
        "업종_대분류",
        "요구불입금금액"
    ]
].copy()

base["연도"] = base["기준월"].dt.year
base["월"] = base["기준월"].dt.month

base["log_입금"] = np.log1p(
    base["요구불입금금액"]
)


# ------------------------------------------------------------
# 1. 기업×연도별 관측수
# ------------------------------------------------------------

fy_info = (
    base
    .groupby(["법인ID", "연도"])
    .agg(
        관측월수=("월", "nunique"),
        연간_log평균=("log_입금", "mean")
    )
    .reset_index()
)

fy_info = fy_info[
    fy_info["관측월수"] >= 6
].copy()


# ------------------------------------------------------------
# 2. 기업×연도 고유수준 제거
# ------------------------------------------------------------

season_idx_base = base.merge(
    fy_info[
        [
            "법인ID",
            "연도",
            "연간_log평균"
        ]
    ],
    on=["법인ID", "연도"],
    how="inner"
)

season_idx_base["월효과_log"] = (
    season_idx_base["log_입금"]
    - season_idx_base["연간_log평균"]
)


# ------------------------------------------------------------
# 3. 연도×월별 중앙효과
# ------------------------------------------------------------

year_month_effect = (
    season_idx_base
    .groupby(["연도", "월"])["월효과_log"]
    .agg(
        기업수="count",
        중앙효과_log="median"
    )
    .reset_index()
)

year_month_effect["효과_pct"] = (
    np.exp(
        year_month_effect["중앙효과_log"]
    ) - 1
) * 100


print("=" * 80)
print("연도별 공통 월효과 (%)")
print("=" * 80)

print(
    year_month_effect
    .pivot(
        index="월",
        columns="연도",
        values="효과_pct"
    )
    .round(2)
)


# ------------------------------------------------------------
# 4. 연도 균등 최종 월효과
#
# 각 연도의 중앙효과를 다시 중앙값
# → 어느 한 연도의 기업수가 많다고 지배하지 않음
# ------------------------------------------------------------

common_month = (
    year_month_effect
    .groupby("월")["중앙효과_log"]
    .median()
    .rename("최종월효과_log")
    .to_frame()
)

common_month["효과_pct"] = (
    np.exp(common_month["최종월효과_log"]) - 1
) * 100


# ------------------------------------------------------------
# 5. 계절지수
#
# exp(log 효과)
# 이후 12개월 기하평균 = 1로 정규화
# ------------------------------------------------------------

common_month["원시계절지수"] = np.exp(
    common_month["최종월효과_log"]
)

geomean = np.exp(
    np.log(
        common_month["원시계절지수"]
    ).mean()
)

common_month["정규화계절지수"] = (
    common_month["원시계절지수"]
    / geomean
)

common_month["조정률_pct"] = (
    common_month["정규화계절지수"] - 1
) * 100


print("\n" + "=" * 80)
print("전체 기업 공통 계절지수 후보")
print("=" * 80)

print(
    common_month[
        [
            "효과_pct",
            "원시계절지수",
            "정규화계절지수",
            "조정률_pct"
        ]
    ]
    .round(4)
)


print("\n12개월 계절지수 기하평균:")
print(
    round(
        np.exp(
            np.log(
                common_month[
                    "정규화계절지수"
                ]
            ).mean()
        ),
        6
    )
)

연도별 공통 월효과 (%)
연도  2023  2024  2025
월                   
1  -0.57  1.29 -2.11
2  -5.73 -1.02 -2.40
3   0.00 -4.50 -0.79
4  -3.90  2.20  0.90
5   1.99  0.00 -1.65
6   0.17 -4.84  0.00
7  -2.11  1.17  0.00
8  -1.18 -4.98 -6.53
9  -2.35 -0.92  1.45
10 -1.26 -3.05 -2.59
11 -0.18 -5.33 -7.84
12  3.46  6.56  9.56

전체 기업 공통 계절지수 후보
    효과_pct  원시계절지수  정규화계절지수  조정률_pct
월                                   
1  -0.5653  0.9943   1.0032   0.3227
2  -2.4002  0.9760   0.9847  -1.5285
3  -0.7889  0.9921   1.0010   0.0971
4   0.8994  1.0090   1.0180   1.8006
5   0.0000  1.0000   1.0089   0.8931
6   0.0000  1.0000   1.0089   0.8931
7   0.0000  1.0000   1.0089   0.8931
8  -4.9755  0.9502   0.9587  -4.1269
9  -0.9203  0.9908   0.9996  -0.0354
10 -2.5876  0.9741   0.9828  -1.7176
11 -5.3315  0.9467   0.9551  -4.4860
12  6.5582  1.0656   1.0751   7.5098

12개월 계절지수 기하평균:
1.0


In [22]:
# ============================================================
# STEP 14. 공통 월효과 Leave-One-Year-Out 검증
#
# 다른 2개 연도만으로 월 지수를 추정한 뒤
# 제외했던 연도에 적용하여 계절효과가 줄어드는지 확인
# ============================================================

focus_months = [2, 8, 10, 11, 12]
years = [2023, 2024, 2025]

loo_common_rows = []

for month in focus_months:

    month_data = (
        year_month_effect[
            year_month_effect["월"] == month
        ]
        .set_index("연도")
    )

    for test_year in years:

        train_years = [
            y for y in years
            if y != test_year
        ]

        # 다른 두 연도의 log 계절효과
        train_effects = (
            month_data.loc[
                train_years,
                "중앙효과_log"
            ]
        )

        # 두 연도의 중앙값
        train_log_effect = (
            train_effects.median()
        )

        # 학습 데이터로 만든 계절지수
        seasonal_index = np.exp(
            train_log_effect
        )

        # 검증연도의 실제 효과
        test_log_effect = (
            month_data.loc[
                test_year,
                "중앙효과_log"
            ]
        )

        before_pct = (
            np.exp(test_log_effect) - 1
        ) * 100

        # 계절효과를 제거한 뒤
        adjusted_log_effect = (
            test_log_effect
            - train_log_effect
        )

        after_pct = (
            np.exp(adjusted_log_effect) - 1
        ) * 100

        loo_common_rows.append({

            "월": month,
            "검증연도": test_year,

            "학습연도":
                ",".join(
                    map(str, train_years)
                ),

            "추정계절지수":
                seasonal_index,

            "조정전효과_pct":
                before_pct,

            "조정후효과_pct":
                after_pct,

            "조정전절대효과":
                abs(before_pct),

            "조정후절대효과":
                abs(after_pct),

            "개선여부":
                abs(after_pct)
                < abs(before_pct)
        })


loo_common = pd.DataFrame(
    loo_common_rows
)


print("=" * 100)
print("월별 Leave-One-Year-Out 상세")
print("=" * 100)

print(
    loo_common
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 월별 검증 성능 요약
# ------------------------------------------------------------

loo_summary = (
    loo_common
    .groupby("월")
    .agg(
        평균_조정전절대효과=(
            "조정전절대효과",
            "mean"
        ),

        평균_조정후절대효과=(
            "조정후절대효과",
            "mean"
        ),

        개선연도수=(
            "개선여부",
            "sum"
        ),

        평균_추정계절지수=(
            "추정계절지수",
            "mean"
        )
    )
)

loo_summary["절대효과_감소율_pct"] = (
    1
    - (
        loo_summary[
            "평균_조정후절대효과"
        ]
        /
        loo_summary[
            "평균_조정전절대효과"
        ]
    )
) * 100


print("\n" + "=" * 100)
print("월별 Leave-One-Year-Out 성능 요약")
print("=" * 100)

print(
    loo_summary
    .round(3)
    .to_string()
)

월별 Leave-One-Year-Out 상세
 월  검증연도      학습연도  추정계절지수  조정전효과_pct  조정후효과_pct  조정전절대효과  조정후절대효과  개선여부
 2  2023 2024,2025   0.983     -5.734     -4.094    5.734    4.094  True
 2  2024 2023,2025   0.959     -1.016      3.196    1.016    3.196 False
 2  2025 2023,2024   0.966     -2.400      1.039    2.400    1.039  True
 8  2023 2024,2025   0.942     -1.185      4.853    1.185    4.853 False
 8  2024 2023,2025   0.961     -4.976     -1.122    4.976    1.122  True
 8  2025 2023,2024   0.969     -6.534     -3.546    6.534    3.546  True
10  2023 2024,2025   0.972     -1.262      1.601    1.262    1.601 False
10  2024 2023,2025   0.981     -3.048     -1.142    3.048    1.142  True
10  2025 2023,2024   0.978     -2.588     -0.438    2.588    0.438  True
11  2023 2024,2025   0.934     -0.181      6.866    0.181    6.866 False
11  2024 2023,2025   0.959     -5.331     -1.298    5.331    1.298  True
11  2025 2023,2024   0.972     -7.839     -5.194    7.839    5.194  True
12  2023 2024,2025   1.080

In [23]:
# ============================================================
# STEP 15. 공통 12월 효과 제거 후 업종별 잔여 12월 효과
# ============================================================

# STEP 13에서 만든 연도별 효과 사용
# 각 연도의 전체 공통 12월 효과를 해당 연도에 제거
common_dec_by_year = (
    year_month_effect[
        year_month_effect["월"] == 12
    ]
    .set_index("연도")["중앙효과_log"]
    .to_dict()
)


resid = season_idx_base.copy()

# 원래 log1p 금액
resid["공통조정_log입금"] = resid["log_입금"]

# 12월만 그 해의 공통 12월 효과 제거
is_dec = resid["월"] == 12

resid.loc[
    is_dec,
    "공통조정_log입금"
] = (
    resid.loc[is_dec, "log_입금"]
    - resid.loc[is_dec, "연도"].map(common_dec_by_year)
)


# ------------------------------------------------------------
# 기업×연도 안에서
# 12월을 제외한 달의 중앙값을 baseline으로 사용
# ------------------------------------------------------------

non_dec = resid[
    resid["월"] != 12
].copy()

baseline = (
    non_dec
    .groupby(["법인ID", "연도"])
    .agg(
        비12월관측수=("월", "nunique"),
        비12월_log중앙값=("공통조정_log입금", "median")
    )
    .reset_index()
)

baseline = baseline[
    baseline["비12월관측수"] >= 6
].copy()


# 12월 자료
dec = resid[
    resid["월"] == 12
][
    [
        "법인ID",
        "업종_대분류",
        "연도",
        "공통조정_log입금"
    ]
].copy()

dec = dec.merge(
    baseline,
    on=["법인ID", "연도"],
    how="inner"
)


# 공통효과를 제거하고도 남은 12월 효과
dec["12월잔여효과_log"] = (
    dec["공통조정_log입금"]
    - dec["비12월_log중앙값"]
)

dec["12월잔여효과_pct"] = (
    np.exp(dec["12월잔여효과_log"]) - 1
) * 100


# ------------------------------------------------------------
# 업종×연도별 잔여효과
# ------------------------------------------------------------

industry_residual = (
    dec
    .groupby(["업종_대분류", "연도"])["12월잔여효과_log"]
    .agg(
        기업수="count",
        중앙잔여효과_log="median"
    )
    .reset_index()
)

industry_residual["중앙잔여효과_pct"] = (
    np.exp(
        industry_residual["중앙잔여효과_log"]
    ) - 1
) * 100


targets = [
    "건설업",
    "전문, 과학 및 기술 서비스업"
]

print("=" * 90)
print("공통 12월 효과 제거 후 업종별 잔여효과")
print("=" * 90)

print(
    industry_residual[
        industry_residual["업종_대분류"].isin(targets)
    ][
        [
            "업종_대분류",
            "연도",
            "기업수",
            "중앙잔여효과_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3개 연도 대표 잔여효과
# 연도별 효과에 동일 가중
# ------------------------------------------------------------

final_industry_residual = (
    industry_residual[
        industry_residual["업종_대분류"].isin(targets)
    ]
    .groupby("업종_대분류")["중앙잔여효과_log"]
    .median()
    .rename("대표잔여효과_log")
    .to_frame()
)

final_industry_residual["대표잔여효과_pct"] = (
    np.exp(
        final_industry_residual["대표잔여효과_log"]
    ) - 1
) * 100

final_industry_residual["추가계절계수"] = (
    np.exp(
        final_industry_residual["대표잔여효과_log"]
    )
)

print("\n" + "=" * 90)
print("업종별 12월 추가계절효과 후보")
print("=" * 90)

print(
    final_industry_residual
    .round(4)
    .to_string()
)

공통 12월 효과 제거 후 업종별 잔여효과
          업종_대분류   연도  기업수  중앙잔여효과_pct
             건설업 2023 1355       54.53
             건설업 2024 1360       45.62
             건설업 2025 1354       54.30
전문, 과학 및 기술 서비스업 2023  251        5.72
전문, 과학 및 기술 서비스업 2024  273       23.99
전문, 과학 및 기술 서비스업 2025  295        2.86

업종별 12월 추가계절효과 후보
                  대표잔여효과_log  대표잔여효과_pct  추가계절계수
업종_대분류                                          
건설업                   0.4337     54.3008  1.5430
전문, 과학 및 기술 서비스업      0.0556      5.7159  1.0572


In [24]:
# ============================================================
# STEP 16. 업종별 12월 추가지수 Leave-One-Year-Out 검증
# ============================================================

targets = [
    "건설업",
    "전문, 과학 및 기술 서비스업"
]

years = [2023, 2024, 2025]

industry_loo_rows = []

for industry in targets:

    temp = industry_residual[
        industry_residual["업종_대분류"] == industry
    ].copy()

    temp = temp.set_index("연도")

    for test_year in years:

        train_years = [
            y for y in years
            if y != test_year
        ]

        # 다른 두 연도의 log 잔여효과로 추가지수 생성
        train_log_effect = (
            temp.loc[
                train_years,
                "중앙잔여효과_log"
            ]
            .median()
        )

        additional_index = np.exp(
            train_log_effect
        )

        # 검증연도의 실제 잔여효과
        test_log_effect = (
            temp.loc[
                test_year,
                "중앙잔여효과_log"
            ]
        )

        before_pct = (
            np.exp(test_log_effect) - 1
        ) * 100

        # 추가 계절효과 제거
        adjusted_log = (
            test_log_effect
            - train_log_effect
        )

        after_pct = (
            np.exp(adjusted_log) - 1
        ) * 100

        industry_loo_rows.append({

            "업종_대분류": industry,

            "검증연도": test_year,

            "학습연도":
                ",".join(
                    map(str, train_years)
                ),

            "추가계절지수":
                additional_index,

            "조정전_잔여효과_pct":
                before_pct,

            "조정후_잔여효과_pct":
                after_pct,

            "조정전절대효과":
                abs(before_pct),

            "조정후절대효과":
                abs(after_pct),

            "개선여부":
                abs(after_pct)
                < abs(before_pct)
        })


industry_loo = pd.DataFrame(
    industry_loo_rows
)


print("=" * 100)
print("업종별 추가계절지수 Leave-One-Year-Out")
print("=" * 100)

print(
    industry_loo
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 업종별 요약
# ------------------------------------------------------------

industry_loo_summary = (
    industry_loo
    .groupby("업종_대분류")
    .agg(
        평균_조정전절대효과=(
            "조정전절대효과",
            "mean"
        ),

        평균_조정후절대효과=(
            "조정후절대효과",
            "mean"
        ),

        개선연도수=(
            "개선여부",
            "sum"
        ),

        평균_추가계절지수=(
            "추가계절지수",
            "mean"
        )
    )
)

industry_loo_summary[
    "절대효과_감소율_pct"
] = (
    1
    - (
        industry_loo_summary[
            "평균_조정후절대효과"
        ]
        /
        industry_loo_summary[
            "평균_조정전절대효과"
        ]
    )
) * 100


print("\n" + "=" * 100)
print("업종별 추가계절지수 검증 요약")
print("=" * 100)

print(
    industry_loo_summary
    .round(3)
    .to_string()
)

업종별 추가계절지수 Leave-One-Year-Out
          업종_대분류  검증연도      학습연도  추가계절지수  조정전_잔여효과_pct  조정후_잔여효과_pct  조정전절대효과  조정후절대효과  개선여부
             건설업  2023 2024,2025   1.499        54.531         3.090   54.531    3.090  True
             건설업  2024 2023,2025   1.544        45.625        -5.693   45.625    5.693  True
             건설업  2025 2023,2024   1.500        54.301         2.859   54.301    2.859  True
전문, 과학 및 기술 서비스업  2023 2024,2025   1.129         5.716        -6.390    5.716    6.390 False
전문, 과학 및 기술 서비스업  2024 2023,2025   1.043        23.991        18.905   23.991   18.905  True
전문, 과학 및 기술 서비스업  2025 2023,2024   1.145         2.859       -10.158    2.859   10.158 False

업종별 추가계절지수 검증 요약
                  평균_조정전절대효과  평균_조정후절대효과  개선연도수  평균_추가계절지수  절대효과_감소율_pct
업종_대분류                                                                  
건설업                   51.486       3.881      3      1.514        92.463
전문, 과학 및 기술 서비스업      10.855      11.818      1      1.106        -8.865


In [25]:
# ============================================================
# STEP 17. 최종 계절조정 요구불입금금액 생성
#
# 공통 조정:
# 8월  0.9502
# 10월 0.9741
# 12월 1.0656
#
# 건설업 12월 추가:
# 1.5430
#
# 그 외 월:
# 조정 없음
# ============================================================

season_adjusted = season_base.copy()

season_adjusted["연도"] = season_adjusted["기준월"].dt.year
season_adjusted["월"] = season_adjusted["기준월"].dt.month


# ------------------------------------------------------------
# 1. 기본 공통 계절지수
# ------------------------------------------------------------

common_index = {
    8: 0.9502,
    10: 0.9741,
    12: 1.0656
}

season_adjusted["공통계절지수"] = (
    season_adjusted["월"]
    .map(common_index)
    .fillna(1.0)
)


# ------------------------------------------------------------
# 2. 업종 추가지수
# ------------------------------------------------------------

season_adjusted["업종추가계절지수"] = 1.0

construction_dec = (
    (season_adjusted["업종_대분류"] == "건설업")
    & (season_adjusted["월"] == 12)
)

season_adjusted.loc[
    construction_dec,
    "업종추가계절지수"
] = 1.5430


# ------------------------------------------------------------
# 3. 최종 총 계절지수
# ------------------------------------------------------------

season_adjusted["총계절지수"] = (
    season_adjusted["공통계절지수"]
    * season_adjusted["업종추가계절지수"]
)


# ------------------------------------------------------------
# 4. log1p 공간에서 정확하게 조정
# ------------------------------------------------------------

season_adjusted["계절조정_요구불입금금액"] = np.expm1(
    np.log1p(
        season_adjusted["요구불입금금액"]
    )
    - np.log(
        season_adjusted["총계절지수"]
    )
)


# 아주 작은 부동소수 오차 방지
season_adjusted[
    "계절조정_요구불입금금액"
] = (
    season_adjusted[
        "계절조정_요구불입금금액"
    ]
    .clip(lower=0)
)


# ------------------------------------------------------------
# 5. 적용 현황 확인
# ------------------------------------------------------------

print("공통지수 적용 행 수:")
print(
    season_adjusted[
        season_adjusted["공통계절지수"] != 1
    ]["월"]
    .value_counts()
    .sort_index()
)

print("\n건설업 12월 추가지수 적용 행 수:")
print(construction_dec.sum())

print("\n총계절지수 종류:")
print(
    season_adjusted["총계절지수"]
    .value_counts()
    .sort_index()
)

공통지수 적용 행 수:
월
8     30959
10    31001
12    30107
Name: count, dtype: int64

건설업 12월 추가지수 적용 행 수:
4355

총계절지수 종류:
총계절지수
0.950200     30959
0.974100     31001
1.000000    273921
1.065600     25752
1.644221      4355
Name: count, dtype: int64


In [26]:
# ============================================================
# STEP 18. 정밀 계수로 최종 계절조정값 생성 + 제거 성능 검증
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. 앞 단계에서 계산한 정밀 계수 사용
# ------------------------------------------------------------

common_index_exact = {
    8: float(common_month.loc[8, "원시계절지수"]),
    10: float(common_month.loc[10, "원시계절지수"]),
    12: float(common_month.loc[12, "원시계절지수"])
}

construction_extra_exact = float(
    final_industry_residual.loc[
        "건설업",
        "추가계절계수"
    ]
)

print("정밀 공통계절지수:")
for m, v in common_index_exact.items():
    print(f"{m}월: {v:.8f}")

print(
    "건설업 12월 추가계절지수:",
    f"{construction_extra_exact:.8f}"
)


# ------------------------------------------------------------
# 2. 최종 조정 데이터
# ------------------------------------------------------------

final_adj = season_base.copy()

final_adj["연도"] = final_adj["기준월"].dt.year
final_adj["월"] = final_adj["기준월"].dt.month

final_adj["공통계절지수"] = (
    final_adj["월"]
    .map(common_index_exact)
    .fillna(1.0)
)

final_adj["업종추가지수"] = 1.0

construction_dec = (
    (final_adj["업종_대분류"] == "건설업")
    & (final_adj["월"] == 12)
)

final_adj.loc[
    construction_dec,
    "업종추가지수"
] = construction_extra_exact

final_adj["총계절지수"] = (
    final_adj["공통계절지수"]
    * final_adj["업종추가지수"]
)


# 실제 분석에 사용할 계절조정 금액
final_adj["계절조정_요구불입금금액"] = (
    final_adj["요구불입금금액"]
    / final_adj["총계절지수"]
)


# ------------------------------------------------------------
# 3. 기본 sanity check
# ------------------------------------------------------------

print("\n음수 계절조정값 수:",
      (final_adj["계절조정_요구불입금금액"] < 0).sum())

print("원래 0이었던 값 중 조정 후 0이 아닌 행 수:",
      (
          (final_adj["요구불입금금액"] == 0)
          &
          (final_adj["계절조정_요구불입금금액"] != 0)
      ).sum())


# ============================================================
# 4. 조정 후 '전체 공통 월효과' 다시 계산
# STEP 13과 동일한 논리
# ============================================================

check = final_adj[
    [
        "법인ID",
        "기준월",
        "업종_대분류",
        "계절조정_요구불입금금액"
    ]
].copy()

check["연도"] = check["기준월"].dt.year
check["월"] = check["기준월"].dt.month

check["log_조정입금"] = np.log1p(
    check["계절조정_요구불입금금액"]
)


# 기업×연도 대표수준
check_fy = (
    check
    .groupby(["법인ID", "연도"])
    .agg(
        관측월수=("월", "nunique"),
        연간_log평균=("log_조정입금", "mean")
    )
    .reset_index()
)

check_fy = check_fy[
    check_fy["관측월수"] >= 6
].copy()


check = check.merge(
    check_fy[
        [
            "법인ID",
            "연도",
            "연간_log평균"
        ]
    ],
    on=["법인ID", "연도"],
    how="inner"
)


check["조정후_월효과_log"] = (
    check["log_조정입금"]
    - check["연간_log평균"]
)


after_year_month = (
    check
    .groupby(["연도", "월"])["조정후_월효과_log"]
    .median()
    .reset_index()
)

after_year_month["조정후효과_pct"] = (
    np.exp(
        after_year_month["조정후_월효과_log"]
    ) - 1
) * 100


# ------------------------------------------------------------
# 5. 조정 전 효과와 결합
# ------------------------------------------------------------

before = year_month_effect[
    ["연도", "월", "효과_pct"]
].rename(
    columns={
        "효과_pct": "조정전효과_pct"
    }
)

validation = before.merge(
    after_year_month[
        [
            "연도",
            "월",
            "조정후효과_pct"
        ]
    ],
    on=["연도", "월"],
    how="inner"
)

validation["조정전절대효과"] = (
    validation["조정전효과_pct"].abs()
)

validation["조정후절대효과"] = (
    validation["조정후효과_pct"].abs()
)


# 핵심 월만
focus = validation[
    validation["월"].isin([8, 10, 12])
].copy()

print("\n" + "=" * 90)
print("공통 계절조정 전후 - 연도별")
print("=" * 90)

print(
    focus[
        [
            "월",
            "연도",
            "조정전효과_pct",
            "조정후효과_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)


# 월별 평균 절대효과 변화
common_validation = (
    focus
    .groupby("월")
    .agg(
        조정전_평균절대효과=(
            "조정전절대효과",
            "mean"
        ),
        조정후_평균절대효과=(
            "조정후절대효과",
            "mean"
        )
    )
)

common_validation["감소율_pct"] = (
    1
    - (
        common_validation[
            "조정후_평균절대효과"
        ]
        /
        common_validation[
            "조정전_평균절대효과"
        ]
    )
) * 100


print("\n" + "=" * 90)
print("공통 계절조정 제거 성능")
print("=" * 90)

print(
    common_validation
    .round(3)
    .to_string()
)


# ============================================================
# 6. 건설업 12월 잔여효과 최종 확인
# ============================================================

construction_check = check[
    check["업종_대분류"] == "건설업"
].copy()


# 비12월을 해당 기업×연도 baseline으로 사용
construction_nondec = (
    construction_check[
        construction_check["월"] != 12
    ]
    .groupby(["법인ID", "연도"])
    .agg(
        비12월관측수=("월", "nunique"),
        비12월_log중앙값=("log_조정입금", "median")
    )
    .reset_index()
)

construction_nondec = construction_nondec[
    construction_nondec["비12월관측수"] >= 6
]


construction_dec_check = (
    construction_check[
        construction_check["월"] == 12
    ][
        [
            "법인ID",
            "연도",
            "log_조정입금"
        ]
    ]
    .merge(
        construction_nondec,
        on=["법인ID", "연도"],
        how="inner"
    )
)


construction_dec_check["잔여효과_log"] = (
    construction_dec_check["log_조정입금"]
    - construction_dec_check["비12월_log중앙값"]
)


construction_final = (
    construction_dec_check
    .groupby("연도")["잔여효과_log"]
    .agg(
        기업수="count",
        중앙잔여효과_log="median"
    )
    .reset_index()
)

construction_final["중앙잔여효과_pct"] = (
    np.exp(
        construction_final["중앙잔여효과_log"]
    ) - 1
) * 100


print("\n" + "=" * 90)
print("건설업 12월 최종 조정 후 잔여효과")
print("=" * 90)

print(
    construction_final[
        [
            "연도",
            "기업수",
            "중앙잔여효과_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

정밀 공통계절지수:
8월: 0.95024452
10월: 0.97412448
12월: 1.06558169
건설업 12월 추가계절지수: 1.54300794

음수 계절조정값 수: 0
원래 0이었던 값 중 조정 후 0이 아닌 행 수: 0

공통 계절조정 전후 - 연도별
 월   연도  조정전효과_pct  조정후효과_pct
 8 2023     -1.185      1.616
10 2023     -1.262      0.000
12 2023      3.461     -2.452
 8 2024     -4.976     -0.804
10 2024     -3.048     -0.786
12 2024      6.558     -0.059
 8 2025     -6.534     -2.435
10 2025     -2.588     -0.338
12 2025      9.555      1.634

공통 계절조정 제거 성능
    조정전_평균절대효과  조정후_평균절대효과  감소율_pct
월                                  
8        4.232       1.618   61.753
10       2.299       0.375   83.701
12       6.525       1.382   78.824

건설업 12월 최종 조정 후 잔여효과
  연도  기업수  중앙잔여효과_pct
2023 1355       0.000
2024 1360       0.000
2025 1354       7.254


In [27]:
# ============================================================
# STEP 19. 악화신호 계산용 36개월 달력 패널 생성
#
# 중요:
# - 보간 사용 안 함
# - 원래 결측은 NaN 유지
# - 2023.01 ~ 2025.12 실제 달력 기준
# - 계절조정 지수 확정값 적용
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. 15,473개 법인 × 36개월 완전 달력 생성
# ------------------------------------------------------------

firm_ids = df["법인ID"].drop_duplicates()

months = pd.date_range(
    "2023-01-01",
    "2025-12-01",
    freq="MS"
)

calendar = pd.MultiIndex.from_product(
    [firm_ids, months],
    names=["법인ID", "기준월"]
).to_frame(index=False)


# ------------------------------------------------------------
# 2. 기업별 업종 연결
# 업종 변경 기업은 앞서 0개였으므로 한 기업당 하나
# ------------------------------------------------------------

firm_industry = (
    df[
        ["법인ID", "업종_대분류"]
    ]
    .dropna(subset=["업종_대분류"])
    .drop_duplicates("법인ID")
)

signal_panel = calendar.merge(
    firm_industry,
    on="법인ID",
    how="left"
)


# ------------------------------------------------------------
# 3. 원래 관측값만 연결
# 보간값은 다시 NaN 처리
# ------------------------------------------------------------

observed = df[
    [
        "법인ID",
        "기준월",
        "요구불입금금액",
        "요구불입금금액_보간여부"
    ]
].copy()

observed["요구불입금_원관측"] = np.where(
    observed["요구불입금금액"].notna()
    & (~observed["요구불입금금액_보간여부"]),
    observed["요구불입금금액"],
    np.nan
)

observed = observed[
    [
        "법인ID",
        "기준월",
        "요구불입금_원관측"
    ]
]

signal_panel = signal_panel.merge(
    observed,
    on=["법인ID", "기준월"],
    how="left"
)


# ------------------------------------------------------------
# 4. 확정 계절지수 적용
# ------------------------------------------------------------

signal_panel["월"] = (
    signal_panel["기준월"].dt.month
)

signal_panel["계절지수"] = 1.0

signal_panel.loc[
    signal_panel["월"] == 8,
    "계절지수"
] = 0.95024452

signal_panel.loc[
    signal_panel["월"] == 10,
    "계절지수"
] = 0.97412448

signal_panel.loc[
    signal_panel["월"] == 12,
    "계절지수"
] = 1.06558169


# 건설업 12월 추가 조정
construction_dec = (
    (signal_panel["업종_대분류"] == "건설업")
    & (signal_panel["월"] == 12)
)

signal_panel.loc[
    construction_dec,
    "계절지수"
] *= 1.54300794


# ------------------------------------------------------------
# 5. 계절조정 금액
# 결측은 그대로 NaN
# ------------------------------------------------------------

signal_panel["요구불입금_계절조정"] = (
    signal_panel["요구불입금_원관측"]
    / signal_panel["계절지수"]
)


# ------------------------------------------------------------
# 6. 구조 확인
# ------------------------------------------------------------

print("전체 달력 행 수:", len(signal_panel))
print("법인 수:", signal_panel["법인ID"].nunique())

print("\n월 범위:")
print(
    signal_panel["기준월"].min(),
    "~",
    signal_panel["기준월"].max()
)

print("\n원관측 존재 행 수:")
print(signal_panel["요구불입금_원관측"].notna().sum())

print("\n원관측 결측 행 수:")
print(signal_panel["요구불입금_원관측"].isna().sum())

print("\n계절조정 후 결측 행 수:")
print(signal_panel["요구불입금_계절조정"].isna().sum())

print("\n기업×월 중복:")
print(
    signal_panel.duplicated(
        ["법인ID", "기준월"]
    ).sum()
)

전체 달력 행 수: 557028
법인 수: 15473

월 범위:
2023-01-01 00:00:00 ~ 2025-12-01 00:00:00

원관측 존재 행 수:
365988

원관측 결측 행 수:
191040

계절조정 후 결측 행 수:
191040

기업×월 중복:
0


In [28]:
# ============================================================
# STEP 20. 계절조정 후 단기 악화신호 계산
#
# 과거: t-5 ~ t-3
# 최근: t-2 ~ t
#
# 조건
# - 6개월 모두 실제 관측
# - 과거 3개월 평균 >= 10백만원
# - 최근 3개월 평균 <= 과거의 70%
# ============================================================

signal_test = (
    signal_panel
    .sort_values(["법인ID", "기준월"])
    .copy()
)


# ------------------------------------------------------------
# 1. 최근 3개월 평균 / 관측수
# ------------------------------------------------------------

signal_test["최근3개월평균"] = (
    signal_test
    .groupby("법인ID")["요구불입금_계절조정"]
    .transform(
        lambda s: s.rolling(
            window=3,
            min_periods=3
        ).mean()
    )
)

signal_test["최근3개월관측수"] = (
    signal_test
    .groupby("법인ID")["요구불입금_계절조정"]
    .transform(
        lambda s: s.rolling(
            window=3,
            min_periods=1
        ).count()
    )
)


# ------------------------------------------------------------
# 2. 과거 3개월 = 최근3개월 값을 3개월 뒤로 이동
#
# t시점 기준:
# 최근 = t-2, t-1, t
# 과거 = t-5, t-4, t-3
# ------------------------------------------------------------

signal_test["과거3개월평균"] = (
    signal_test
    .groupby("법인ID")["최근3개월평균"]
    .shift(3)
)

signal_test["과거3개월관측수"] = (
    signal_test
    .groupby("법인ID")["최근3개월관측수"]
    .shift(3)
)


# ------------------------------------------------------------
# 3. 6개월 완전관측 여부
# ------------------------------------------------------------

signal_test["6개월완전관측"] = (
    (signal_test["과거3개월관측수"] == 3)
    &
    (signal_test["최근3개월관측수"] == 3)
)


# ------------------------------------------------------------
# 4. 변화율
# 과거평균이 최소 10백만원인 경우에만 계산
# ------------------------------------------------------------

BASELINE_MIN = 10.0
DECLINE_RATIO = 0.70

valid_baseline = (
    signal_test["6개월완전관측"]
    &
    (signal_test["과거3개월평균"] >= BASELINE_MIN)
)

signal_test["최근대비비율"] = np.nan

signal_test.loc[
    valid_baseline,
    "최근대비비율"
] = (
    signal_test.loc[
        valid_baseline,
        "최근3개월평균"
    ]
    /
    signal_test.loc[
        valid_baseline,
        "과거3개월평균"
    ]
)

signal_test["감소율_pct"] = (
    1 - signal_test["최근대비비율"]
) * 100


# ------------------------------------------------------------
# 5. 새 악화신호
# ------------------------------------------------------------

signal_test["단기악화신호"] = (
    valid_baseline
    &
    (
        signal_test["최근대비비율"]
        <= DECLINE_RATIO
    )
)


# ============================================================
# 6. 전체 결과 확인
# ============================================================

eligible = signal_test[
    valid_baseline
]

signals = signal_test[
    signal_test["단기악화신호"]
]


print("=" * 90)
print("계절조정 단기 악화신호")
print("=" * 90)

print("6개월 완전관측 건수:",
      signal_test["6개월완전관측"].sum())

print("기준예치 10 이상 포함 판정가능 건수:",
      len(eligible))

print("악화신호 건수:",
      len(signals))

print(
    "악화신호 비율:",
    round(
        len(signals) / len(eligible) * 100,
        2
    ),
    "%"
)


print("\n악화신호 발생 기업 수:",
      signals["법인ID"].nunique())


# ------------------------------------------------------------
# 7. 월별 결과
# ------------------------------------------------------------

monthly_signal = (
    signal_test[
        valid_baseline
    ]
    .groupby("기준월")
    .agg(
        판정가능건수=("법인ID", "size"),
        악화신호건수=("단기악화신호", "sum")
    )
)

monthly_signal["악화신호율_pct"] = (
    monthly_signal["악화신호건수"]
    /
    monthly_signal["판정가능건수"]
    * 100
)


print("\n" + "=" * 90)
print("월별 악화신호")
print("=" * 90)

print(
    monthly_signal
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 8. 기존 YoY 신호와 동일한 평가기간도 별도 확인
#
# 기존 신호 기간:
# 2024-03 ~ 2025-06
# ------------------------------------------------------------

comparison_period = signal_test[
    signal_test["기준월"].between(
        "2024-03-01",
        "2025-06-01"
    )
]

comparison_eligible = comparison_period[
    comparison_period["6개월완전관측"]
    &
    (comparison_period["과거3개월평균"] >= BASELINE_MIN)
]

comparison_signals = comparison_period[
    comparison_period["단기악화신호"]
]


print("\n" + "=" * 90)
print("기존 YoY와 비교 가능한 기간: 2024-03 ~ 2025-06")
print("=" * 90)

print(
    "판정가능 건수:",
    len(comparison_eligible)
)

print(
    "악화신호 건수:",
    len(comparison_signals)
)

print(
    "악화신호 발생 기업 수:",
    comparison_signals["법인ID"].nunique()
)

print(
    "악화신호율:",
    round(
        len(comparison_signals)
        / len(comparison_eligible)
        * 100,
        2
    ),
    "%"
)

계절조정 단기 악화신호
6개월 완전관측 건수: 252173
기준예치 10 이상 포함 판정가능 건수: 192295
악화신호 건수: 45401
악화신호 비율: 23.61 %

악화신호 발생 기업 수: 9043

월별 악화신호
            판정가능건수  악화신호건수  악화신호율_pct
기준월                                  
2023-06-01    6260    1306      20.86
2023-07-01    6239    1249      20.02
2023-08-01    6272    1419      22.62
2023-09-01    6244    1478      23.67
2023-10-01    6301    1448      22.98
2023-11-01    6347    1465      23.08
2023-12-01    6304    1390      22.05
2024-01-01    6323    1258      19.90
2024-02-01    6275    1349      21.50
2024-03-01    6260    1731      27.65
2024-04-01    6240    1643      26.33
2024-05-01    6147    1517      24.68
2024-06-01    6008    1313      21.85
2024-07-01    6021    1350      22.42
2024-08-01    6063    1415      23.34
2024-09-01    6150    1467      23.85
2024-10-01    6214    1477      23.77
2024-11-01    6253    1486      23.76
2024-12-01    6339    1482      23.38
2025-01-01    6367    1498      23.53
2025-02-01    6295    1523      24.19
20

In [29]:
# ============================================================
# STEP 21. 계절조정 전 vs 후
# 동일한 3개월 vs 3개월 악화신호 직접 비교
# ============================================================

import pandas as pd
import numpy as np


BASELINE_MIN = 10.0
DECLINE_RATIO = 0.70


def make_short_signal(panel, value_col, label):

    x = (
        panel[
            [
                "법인ID",
                "기준월",
                value_col
            ]
        ]
        .sort_values(
            ["법인ID", "기준월"]
        )
        .copy()
    )

    # ----------------------------------------
    # 최근 t-2 ~ t
    # ----------------------------------------

    x[f"{label}_최근3평균"] = (
        x
        .groupby("법인ID")[value_col]
        .transform(
            lambda s:
            s.rolling(
                3,
                min_periods=3
            ).mean()
        )
    )

    x[f"{label}_최근3관측"] = (
        x
        .groupby("법인ID")[value_col]
        .transform(
            lambda s:
            s.rolling(
                3,
                min_periods=1
            ).count()
        )
    )

    # ----------------------------------------
    # 과거 t-5 ~ t-3
    # ----------------------------------------

    x[f"{label}_과거3평균"] = (
        x
        .groupby("법인ID")[
            f"{label}_최근3평균"
        ]
        .shift(3)
    )

    x[f"{label}_과거3관측"] = (
        x
        .groupby("법인ID")[
            f"{label}_최근3관측"
        ]
        .shift(3)
    )

    # ----------------------------------------
    # 판정 가능
    # ----------------------------------------

    x[f"{label}_판정가능"] = (
        (x[f"{label}_최근3관측"] == 3)
        &
        (x[f"{label}_과거3관측"] == 3)
        &
        (
            x[f"{label}_과거3평균"]
            >= BASELINE_MIN
        )
    )

    # ----------------------------------------
    # 비율
    # ----------------------------------------

    x[f"{label}_비율"] = np.nan

    mask = x[f"{label}_판정가능"]

    x.loc[
        mask,
        f"{label}_비율"
    ] = (
        x.loc[
            mask,
            f"{label}_최근3평균"
        ]
        /
        x.loc[
            mask,
            f"{label}_과거3평균"
        ]
    )

    # ----------------------------------------
    # 신호
    # ----------------------------------------

    x[f"{label}_신호"] = (
        x[f"{label}_판정가능"]
        &
        (
            x[f"{label}_비율"]
            <= DECLINE_RATIO
        )
    )

    return x[
        [
            "법인ID",
            "기준월",
            f"{label}_판정가능",
            f"{label}_비율",
            f"{label}_신호"
        ]
    ]


# ============================================================
# 1. 조정 전
# ============================================================

raw_signal = make_short_signal(
    signal_panel,
    "요구불입금_원관측",
    "조정전"
)


# ============================================================
# 2. 조정 후
# ============================================================

adj_signal = make_short_signal(
    signal_panel,
    "요구불입금_계절조정",
    "조정후"
)


# ============================================================
# 3. 결합
# ============================================================

compare_signal = raw_signal.merge(
    adj_signal,
    on=["법인ID", "기준월"],
    how="inner"
)


# ------------------------------------------------------------
# 각 방식 자체 판정가능 표본 기준 월별 신호율
# ------------------------------------------------------------

def monthly_rate(
    data,
    eligible_col,
    signal_col
):

    temp = data[
        data[eligible_col]
    ]

    out = (
        temp
        .groupby("기준월")
        .agg(
            판정가능건수=(
                eligible_col,
                "size"
            ),
            신호건수=(
                signal_col,
                "sum"
            )
        )
    )

    out["신호율_pct"] = (
        out["신호건수"]
        /
        out["판정가능건수"]
        * 100
    )

    return out


raw_month = monthly_rate(
    compare_signal,
    "조정전_판정가능",
    "조정전_신호"
)

adj_month = monthly_rate(
    compare_signal,
    "조정후_판정가능",
    "조정후_신호"
)


monthly_compare = (
    raw_month[
        ["신호율_pct"]
    ]
    .rename(
        columns={
            "신호율_pct":
            "조정전_신호율_pct"
        }
    )
    .join(
        adj_month[
            ["신호율_pct"]
        ].rename(
            columns={
                "신호율_pct":
                "조정후_신호율_pct"
            }
        ),
        how="outer"
    )
)


monthly_compare["변화_pp"] = (
    monthly_compare[
        "조정후_신호율_pct"
    ]
    -
    monthly_compare[
        "조정전_신호율_pct"
    ]
)


print("=" * 90)
print("계절조정 전후 월별 단기 악화신호율")
print("=" * 90)

print(
    monthly_compare
    .round(2)
    .to_string()
)


# ============================================================
# 4. 두 방식 모두 판정가능한 동일 표본만 비교
# ============================================================

common_eligible = compare_signal[
    compare_signal["조정전_판정가능"]
    &
    compare_signal["조정후_판정가능"]
].copy()


common_month = (
    common_eligible
    .groupby("기준월")
    .agg(
        동일판정표본=("법인ID", "size"),
        조정전신호=("조정전_신호", "sum"),
        조정후신호=("조정후_신호", "sum")
    )
)

common_month["조정전율_pct"] = (
    common_month["조정전신호"]
    /
    common_month["동일판정표본"]
    * 100
)

common_month["조정후율_pct"] = (
    common_month["조정후신호"]
    /
    common_month["동일판정표본"]
    * 100
)

common_month["변화_pp"] = (
    common_month["조정후율_pct"]
    -
    common_month["조정전율_pct"]
)


print("\n" + "=" * 90)
print("동일 판정가능 표본 기준 비교")
print("=" * 90)

print(
    common_month[
        [
            "동일판정표본",
            "조정전율_pct",
            "조정후율_pct",
            "변화_pp"
        ]
    ]
    .round(2)
    .to_string()
)


# ============================================================
# 5. 계절조정 때문에 실제 판정이 바뀐 건수
# ============================================================

common_eligible["신호변화"] = np.select(

    [
        (
            common_eligible["조정전_신호"]
            &
            ~common_eligible["조정후_신호"]
        ),

        (
            ~common_eligible["조정전_신호"]
            &
            common_eligible["조정후_신호"]
        )
    ],

    [
        "조정후_해제",
        "조정후_신규"
    ],

    default="동일"
)


change_summary = (
    common_eligible[
        common_eligible["신호변화"] != "동일"
    ]
    .groupby(
        ["기준월", "신호변화"]
    )
    .size()
    .unstack(fill_value=0)
)


print("\n" + "=" * 90)
print("계절조정으로 신호 판정이 실제 변경된 건수")
print("=" * 90)

print(
    change_summary.to_string()
)

계절조정 전후 월별 단기 악화신호율
            조정전_신호율_pct  조정후_신호율_pct  변화_pp
기준월                                        
2023-06-01        20.86        20.86   0.00
2023-07-01        20.02        20.02   0.00
2023-08-01        23.10        22.62  -0.48
2023-09-01        24.20        23.67  -0.53
2023-10-01        24.33        22.98  -1.35
2023-11-01        22.67        23.08   0.41
2023-12-01        19.27        22.05   2.78
2024-01-01        17.38        19.90   2.51
2024-02-01        18.82        21.50   2.67
2024-03-01        29.86        27.65  -2.21
2024-04-01        28.85        26.33  -2.52
2024-05-01        27.39        24.68  -2.71
2024-06-01        21.85        21.85   0.00
2024-07-01        22.42        22.42   0.00
2024-08-01        24.20        23.34  -0.86
2024-09-01        24.65        23.85  -0.80
2024-10-01        25.13        23.77  -1.36
2024-11-01        23.44        23.76   0.32
2024-12-01        20.72        23.38   2.66
2025-01-01        20.47        23.53   3.06
2025-02-01  

In [30]:
# ============================================================
# STEP 22. 2월 계절조정 추가 민감도 실험
#
# 현재안:
# 8 / 10 / 12월 + 건설업 12월
#
# 실험안:
# 현재안 + 2월
#
# 목적:
# 2월 보정이 실제 3개월 vs 3개월 악화신호의
# 계절왜곡을 줄이는지 확인
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 2월 정밀 계절지수 재계산
#
# 앞에서 common_month 변수가 다른 용도로 덮였을 수 있으므로
# year_month_effect에서 직접 다시 계산
# ------------------------------------------------------------

feb_log_effect = (
    year_month_effect.loc[
        year_month_effect["월"] == 2,
        "중앙효과_log"
    ]
    .median()
)

feb_index_exact = np.exp(
    feb_log_effect
)

print(
    "2월 정밀 계절지수:",
    round(feb_index_exact, 8)
)

print(
    "2월 대표 계절효과:",
    round(
        (feb_index_exact - 1) * 100,
        3
    ),
    "%"
)


# ------------------------------------------------------------
# 2. 현재 조정값을 복사한 뒤 2월만 추가 조정
# ------------------------------------------------------------

feb_test_panel = signal_panel.copy()

feb_test_panel[
    "요구불입금_계절조정_2월포함"
] = (
    feb_test_panel[
        "요구불입금_계절조정"
    ]
)

feb_mask = (
    feb_test_panel["기준월"].dt.month == 2
)

feb_test_panel.loc[
    feb_mask,
    "요구불입금_계절조정_2월포함"
] = (
    feb_test_panel.loc[
        feb_mask,
        "요구불입금_계절조정"
    ]
    / feb_index_exact
)


# ------------------------------------------------------------
# 3. 기존안 신호
# ------------------------------------------------------------

current_signal = make_short_signal(
    feb_test_panel,
    "요구불입금_계절조정",
    "현재안"
)


# ------------------------------------------------------------
# 4. 2월 포함안 신호
# ------------------------------------------------------------

feb_signal = make_short_signal(
    feb_test_panel,
    "요구불입금_계절조정_2월포함",
    "2월포함"
)


# ------------------------------------------------------------
# 5. 두 결과 결합
# ------------------------------------------------------------

feb_compare = current_signal.merge(
    feb_signal,
    on=["법인ID", "기준월"],
    how="inner"
)


# 두 방식 모두 판정 가능한 동일표본
same = feb_compare[
    feb_compare["현재안_판정가능"]
    &
    feb_compare["2월포함_판정가능"]
].copy()


monthly_feb_compare = (
    same
    .groupby("기준월")
    .agg(
        동일표본=("법인ID", "size"),
        현재안신호=("현재안_신호", "sum"),
        이월포함신호=("2월포함_신호", "sum")
    )
)

monthly_feb_compare["현재안율_pct"] = (
    monthly_feb_compare["현재안신호"]
    / monthly_feb_compare["동일표본"]
    * 100
)

monthly_feb_compare["2월포함율_pct"] = (
    monthly_feb_compare["이월포함신호"]
    / monthly_feb_compare["동일표본"]
    * 100
)

monthly_feb_compare["변화_pp"] = (
    monthly_feb_compare["2월포함율_pct"]
    - monthly_feb_compare["현재안율_pct"]
)


# ------------------------------------------------------------
# 6. 2월의 영향을 받는 시기만 출력
#
# 2월이 최근창: 2~4월
# 2월이 과거창: 5~7월
# ------------------------------------------------------------

affected = monthly_feb_compare[
    monthly_feb_compare.index.month.isin(
        [2, 3, 4, 5, 6, 7]
    )
]


print("\n" + "=" * 90)
print("2월 조정 추가 전후 - 영향을 받는 월")
print("=" * 90)

print(
    affected[
        [
            "동일표본",
            "현재안율_pct",
            "2월포함율_pct",
            "변화_pp"
        ]
    ]
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 7. 연도별 3~4월 평균과 5~7월 평균
# ------------------------------------------------------------

tmp = monthly_feb_compare.reset_index()

tmp["연도"] = tmp["기준월"].dt.year
tmp["월"] = tmp["기준월"].dt.month


spring_summary = (
    tmp[
        tmp["월"].isin([3, 4])
    ]
    .groupby("연도")
    .agg(
        현재안_3_4월평균=(
            "현재안율_pct",
            "mean"
        ),
        이월포함_3_4월평균=(
            "2월포함율_pct",
            "mean"
        )
    )
)


later_summary = (
    tmp[
        tmp["월"].isin([5, 6, 7])
    ]
    .groupby("연도")
    .agg(
        현재안_5_7월평균=(
            "현재안율_pct",
            "mean"
        ),
        이월포함_5_7월평균=(
            "2월포함율_pct",
            "mean"
        )
    )
)


print("\n" + "=" * 90)
print("3~4월 피크 변화")
print("=" * 90)
print(
    spring_summary
    .round(2)
    .to_string()
)


print("\n" + "=" * 90)
print("5~7월 반대방향 영향")
print("=" * 90)
print(
    later_summary
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 8. 실제 판정 변경 건수
# ------------------------------------------------------------

same["변경유형"] = np.select(
    [
        (
            same["현재안_신호"]
            &
            ~same["2월포함_신호"]
        ),
        (
            ~same["현재안_신호"]
            &
            same["2월포함_신호"]
        )
    ],
    [
        "2월조정후_해제",
        "2월조정후_신규"
    ],
    default="동일"
)


feb_changes = (
    same[
        same["변경유형"] != "동일"
    ]
    .groupby(
        ["기준월", "변경유형"]
    )
    .size()
    .unstack(fill_value=0)
)


print("\n" + "=" * 90)
print("2월 추가조정으로 판정이 변경된 건수")
print("=" * 90)

print(
    feb_changes.to_string()
)

2월 정밀 계절지수: 0.97599829
2월 대표 계절효과: -2.4 %

2월 조정 추가 전후 - 영향을 받는 월
            동일표본  현재안율_pct  2월포함율_pct  변화_pp
기준월                                         
2023-06-01  6260     20.86      21.17   0.30
2023-07-01  6239     20.02      20.36   0.34
2024-02-01  6275     21.50      21.15  -0.35
2024-03-01  6260     27.65      27.19  -0.46
2024-04-01  6240     26.33      25.95  -0.38
2024-05-01  6147     24.68      25.09   0.41
2024-06-01  6008     21.85      22.20   0.35
2024-07-01  6021     22.42      22.72   0.30
2025-02-01  6294     24.20      23.83  -0.37
2025-03-01  6295     30.41      29.85  -0.56
2025-04-01  6233     27.60      27.16  -0.43
2025-05-01  6171     25.78      26.12   0.34
2025-06-01  6048     22.45      22.75   0.30
2025-07-01  6072     23.40      23.75   0.35

3~4월 피크 변화
      현재안_3_4월평균  이월포함_3_4월평균
연도                           
2024       26.99        26.57
2025       29.00        28.51

5~7월 반대방향 영향
      현재안_5_7월평균  이월포함_5_7월평균
연도                           
2023    